# Leakage-aware evaluation of a cross-fitted MI-CNN–XGBoost framework on ToN-IoT
Reproduces every table and figure of the manuscript *"Do Deep Features Really Help Gradient Boosting? ..."* (M. R. Jasim).

**Setup on Kaggle**
1. *Add Input* → paste `https://www.kaggle.com/datasets/quocanhnguyeeeeen/train-test-network` (or any copy of `Train_Test_Network.csv` with 461,043 rows).
2. Settings → Internet: on. Accelerator is optional (CPU is sufficient; GPU only speeds up the CNN).
3. *Save Version → Save & Run All (Commit)*. The full grid takes about 3.5–4 h on Kaggle CPU.

The source-host-disjoint folds of protocol P4 are shipped as `p4_host_folds.json`, because `StratifiedGroupKFold(shuffle=True)` in older scikit-learn releases (e.g. 1.6, the current Kaggle image) produces degenerate folds.


In [ ]:
!pip -q install xgboost lightgbm shap 2>/dev/null
import xgboost, lightgbm, shap, tensorflow as tf, sklearn
print(xgboost.__version__, lightgbm.__version__, shap.__version__, tf.__version__, sklearn.__version__)

In [ ]:
import glob, os
cands = [p for p in glob.glob('/kaggle/input/**/*.csv', recursive=True) if 'train_test_network' in os.path.basename(p).lower()]
assert cands, 'Add a ToN-IoT dataset containing Train_Test_Network.csv'
os.environ['TON_CSV'] = cands[0]
os.environ['RESULTS_DIR'] = '/kaggle/working/results'; os.environ['FIG_DIR'] = '/kaggle/working/fig'; os.environ['AGG_OUT'] = '/kaggle/working/agg.json'
os.makedirs(os.environ['RESULTS_DIR'], exist_ok=True)
import pandas as pd; n = pd.read_csv(cands[0], usecols=['label']).shape[0]; print(cands[0], n)
assert n == 461043, 'Expected the 461,043-row Train_Test_Network subset'

## Source code and fixed P4 fold map (written to the working directory)

In [ ]:
%%writefile common.py
"""Shared data preparation, protocols, feature selection and models.
Leakage-aware evaluation of MI-guided hybrid 1D-CNN + XGBoost on ToN-IoT (Train_Test_Network)."""
import os, time, json, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

DATASET = os.environ.get("DATASET", "ton")
LABELS = ['label', 'type']
if DATASET == 'edge':
    # Edge-IIoTset (ML subset). Identifier/payload columns dropped following the dataset authors' preprocessing.
    DATA = os.environ.get("EDGE_CSV", "/home/claude/exp2/edge.csv")
    IDS = ['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.src.proto_ipv4', 'arp.dst.proto_ipv4', 'http.file_data',
           'http.request.full_uri', 'icmp.transmit_timestamp', 'http.request.uri.query', 'tcp.options', 'tcp.payload',
           'tcp.srcport', 'tcp.dstport', 'udp.port', 'mqtt.msg', 'src_ip']
    CLASSES = ['normal', 'backdoor', 'ddos_http', 'ddos_icmp', 'ddos_tcp', 'ddos_udp', 'fingerprinting', 'mitm', 'password',
               'port_scanning', 'ransomware', 'sql_injection', 'uploading', 'vulnerability_scanner', 'xss']
else:
    DATA = os.environ.get("TON_CSV", "/home/claude/exp/ton.csv")
    IDS = ['ts', 'src_ip', 'src_port', 'dst_ip', 'dst_port']
    CLASSES = ['normal', 'backdoor', 'ddos', 'dos', 'injection', 'mitm', 'password', 'ransomware', 'scanning', 'xss']
DROP_EXTRA = [c for c in os.environ.get("DROP_FEATURES", "").split(',') if c]


def load_raw():
    df = pd.read_csv(DATA, low_memory=False)
    if DATASET == 'edge':
        df = df.rename(columns={'Attack_label': 'label', 'Attack_type': 'type'})
        df['type'] = df['type'].astype(str).str.strip().str.lower()
        df['src_ip'] = df['ip.src_host'].astype(str)
    for c in df.columns:
        if df[c].dtype == object or str(df[c].dtype).startswith('str'):
            df[c] = df[c].astype(str)
    return df


def feature_cols(df):
    return [c for c in df.columns if c not in IDS + LABELS + DROP_EXTRA]


def dedup_feature_space(df):
    """Remove exact duplicates in the model's feature space and vectors with conflicting labels."""
    feat = feature_cols(df)
    key = pd.util.hash_pandas_object(df[feat], index=False)
    n_types = df.groupby(key)['type'].transform('nunique')
    clean = df[n_types == 1].copy()
    clean['_key'] = key[n_types == 1].values
    clean = clean.drop_duplicates('_key').drop(columns='_key')
    return clean.reset_index(drop=True), int((n_types > 1).sum())


# ---------------------------------------------------------------- protocols
def split_protocol(df, protocol, seed, test_size=0.2):
    """Return (train_idx, test_idx) for one of four evaluation protocols.
    P1 random  : record-level stratified random split on raw data (common practice)
    P2 flow    : 5-tuple GroupShuffleSplit after record-level dedup (original manuscript)
    P3 dedup   : stratified random split after feature-space dedup
    P4 host    : StratifiedGroupKFold by source host after feature-space dedup (strict)"""
    from sklearn.model_selection import train_test_split, GroupShuffleSplit, StratifiedGroupKFold
    idx = np.arange(len(df))
    if protocol in ('P1', 'P3'):
        tr, te = train_test_split(idx, test_size=test_size, stratify=df['type'], random_state=seed)
    elif protocol == 'P2':
        g = df['src_ip'] + '|' + df['src_port'].astype(str) + '|' + df['dst_ip'] + '|' + df['dst_port'].astype(str) + '|' + df['proto']
        tr, te = next(GroupShuffleSplit(1, test_size=test_size, random_state=seed).split(idx, groups=g))
    elif protocol == 'P6':
        # class-stratified chronological split: within every class, the earliest (1 - test_size) of the
        # flows (by timestamp) are used for training and the latest test_size for testing
        tr, te = [], []
        for _, g in df.groupby('type'):
            o = g.sort_values('ts', kind='mergesort').index.values
            cut = int(round(len(o) * (1 - test_size)))
            tr.extend(o[:cut]); te.extend(o[cut:])
        tr, te = np.sort(tr), np.sort(te)
    elif protocol == 'P4':
        # Fixed source-host -> fold map (generated once with StratifiedGroupKFold(5, shuffle=True, random_state=0),
        # scikit-learn 1.9). Older scikit-learn releases (e.g. 1.6) produce degenerate folds with shuffle=True,
        # so the map is shipped with the code to make P4 identical across environments.
        mp = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'p4_host_folds.json')
        if os.path.exists(mp):
            fold = df['src_ip'].map(json.load(open(mp))).values
            assert not pd.isna(fold).any(), 'unknown source host in P4 fold map'
            te = idx[fold == seed % 5]; tr = idx[fold != seed % 5]
        else:
            sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
            folds = list(sgkf.split(idx, df['label'], groups=df['src_ip']))
            tr, te = folds[seed % 5]
    return np.asarray(tr), np.asarray(te)


def data_for_protocol(protocol):
    raw = load_raw()
    if protocol == 'P1':
        return raw
    if protocol == 'P2':
        return raw.drop_duplicates().reset_index(drop=True)
    d, _ = dedup_feature_space(raw)
    mn = int(os.environ.get('MIN_CLASS_N', '0'))
    if mn:
        vc = d['type'].value_counts(); d = d[d['type'].isin(vc[vc >= mn].index)].reset_index(drop=True)
    return d


# ---------------------------------------------------------------- encoding (fit on train only)
class Encoder:
    """Median imputation for numerics; one-hot for low-cardinality categoricals;
    train-frequency encoding for high-cardinality categoricals (dns_query, http_uri, ...)."""
    def __init__(self, max_ohe=20):
        self.max_ohe = max_ohe

    def fit(self, X):
        self.num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
        self.cat = [c for c in X.columns if c not in self.num]
        self.med = X[self.num].median()
        self.ohe, self.freq = {}, {}
        for c in self.cat:
            vc = X[c].value_counts()
            if len(vc) <= self.max_ohe:
                self.ohe[c] = list(vc.index)
            else:
                self.freq[c] = (vc / len(X)).to_dict()
        return self

    def transform(self, X):
        out = {c: X[c].fillna(self.med[c]).astype(float).values for c in self.num}
        for c, levels in self.ohe.items():
            for lv in levels:
                out[f"{c}={lv}"] = (X[c].values == lv).astype(float)
        for c, m in self.freq.items():
            out[f"{c}_freq"] = X[c].map(m).fillna(0.0).astype(float).values
        F = pd.DataFrame(out)
        self.columns_ = list(F.columns)
        return F

    def parent(self, col):
        return col.split('=')[0].replace('_freq', '')


# ---------------------------------------------------------------- MI + geometric elbow
def mi_elbow(Xtr, ytr, enc, seed=42, n_sub=60000):
    """MI per encoded column (train only), aggregated to original attribute by max, then knee by
    maximum perpendicular distance to the chord (Kneedle-style). Returns selected encoded columns."""
    from sklearn.feature_selection import mutual_info_classif
    rng = np.random.RandomState(seed)
    sub = rng.choice(len(Xtr), min(n_sub, len(Xtr)), replace=False)
    Xs = Xtr.iloc[sub]
    discrete = np.array([('=' in c) for c in Xs.columns])
    mi = mutual_info_classif(Xs.values, ytr[sub], discrete_features=discrete, random_state=seed)
    mi = pd.Series(mi, index=Xs.columns)
    parent = mi.groupby([enc.parent(c) for c in mi.index]).max().sort_values(ascending=False)
    y = parent.values; x = np.arange(1, len(y) + 1)
    num = np.abs((y[-1] - y[0]) * x - (x[-1] - x[0]) * y + x[-1] * y[0] - y[-1] * x[0])
    d = num / np.sqrt((y[-1] - y[0]) ** 2 + (x[-1] - x[0]) ** 2)
    k = int(np.argmax(d)) + 1
    keep_parents = set(parent.index[:k])
    cols = [c for c in Xtr.columns if enc.parent(c) in keep_parents]
    return cols, parent, k


# ---------------------------------------------------------------- models
def xgb_model(n_classes, seed, spw=1.0):
    from xgboost import XGBClassifier
    kw = dict(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, reg_lambda=1.0, tree_method='hist', n_jobs=2, random_state=seed, verbosity=0)
    if n_classes == 2:
        return XGBClassifier(objective='binary:logistic', eval_metric='logloss', scale_pos_weight=spw, **kw)
    return XGBClassifier(objective='multi:softprob', eval_metric='mlogloss', **kw)


def build_cnn(k, n_classes, seed, l2=1e-4):
    import tensorflow as tf
    from tensorflow.keras import layers, regularizers, Model
    tf.keras.utils.set_random_seed(seed)
    inp = layers.Input((k, 1))
    h = inp
    for f in (32, 64):
        h = layers.Conv1D(f, 3, padding='same', kernel_regularizer=regularizers.l2(l2))(h)
        h = layers.BatchNormalization()(h); h = layers.ReLU()(h)
        h = layers.MaxPooling1D(2, padding='same')(h)
    h = layers.Flatten()(h)
    h = layers.Dense(64, activation='relu')(h); h = layers.Dropout(0.3)(h)
    z = layers.Dense(32, activation='relu', name='deep_features')(h)
    out = layers.Dense(1, activation='sigmoid')(z) if n_classes == 2 else layers.Dense(n_classes, activation='softmax')(z)
    m = Model(inp, out)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy' if n_classes == 2 else 'sparse_categorical_crossentropy')
    return m


def fit_cnn(X, y, n_classes, seed, epochs=20, batch=1024, class_weight=None):
    import tensorflow as tf
    from sklearn.model_selection import train_test_split
    tr, va = train_test_split(np.arange(len(X)), test_size=0.1, random_state=seed, stratify=y)
    m = build_cnn(X.shape[1], n_classes, seed)
    cb = [tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
          tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=2, min_lr=1e-5)]
    h = m.fit(X[tr, :, None], y[tr], validation_data=(X[va, :, None], y[va]), epochs=epochs, batch_size=batch,
              callbacks=cb, verbose=0, class_weight=class_weight)
    return m, h.history


def cnn_embed(m, X):
    import tensorflow as tf
    f = tf.keras.Model(m.input, m.get_layer('deep_features').output)
    return f.predict(X[:, :, None], batch_size=4096, verbose=0)


def cnn_proba(m, X):
    p = m.predict(X[:, :, None], batch_size=4096, verbose=0)
    return np.c_[1 - p[:, 0], p[:, 0]] if p.shape[1] == 1 else p


# ---------------------------------------------------------------- metrics
def metrics(y, proba, n_classes):
    from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                                 matthews_corrcoef, roc_auc_score, average_precision_score, confusion_matrix)
    pred = proba.argmax(1)
    r = dict(acc=accuracy_score(y, pred), mcc=matthews_corrcoef(y, pred),
             f1_macro=f1_score(y, pred, average='macro'))
    if n_classes == 2:
        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
        r.update(precision=precision_score(y, pred), recall=recall_score(y, pred), f1=f1_score(y, pred),
                 fpr=fp / (fp + tn), auc=roc_auc_score(y, proba[:, 1]), pr_auc=average_precision_score(y, proba[:, 1]),
                 tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))
    else:
        r['per_class_f1'] = f1_score(y, pred, average=None, labels=list(range(n_classes))).tolist()
        r['recall_macro'] = recall_score(y, pred, average='macro')
    return {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in r.items()}


# ---------------------------------------------------------------- v2 components (host-aware)
def host_folds(y, groups, n=3):
    """Stratified folds that keep every source host in a single fold (host-grouped cross-fitting).
    Falls back to ordinary stratified folds if a training part would miss a class."""
    from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
    idx = np.arange(len(y))
    folds = list(StratifiedGroupKFold(n_splits=n).split(idx, y, groups=groups))
    if all(len(np.unique(y[a])) == len(np.unique(y)) for a, _ in folds):
        return folds, 'host'
    return list(StratifiedKFold(n, shuffle=True, random_state=0).split(idx, y)), 'record'


def mi_classwise_stable(Xtr, types, groups, enc, seed=42, n_sub=30000, n_groups=3):
    """Class-wise, host-stable MI selection. For every class (one-vs-rest) the MI of each encoded column is
    computed separately inside n_groups disjoint host groups; the stable score is the minimum over the groups
    in which the class occurs. Scores are aggregated to the original attribute by max, a geometric knee is taken
    per class, and the union of the per-class selections is returned."""
    from sklearn.feature_selection import mutual_info_classif
    from sklearn.model_selection import GroupKFold
    rng = np.random.RandomState(seed)
    sub = rng.choice(len(Xtr), min(n_sub, len(Xtr)), replace=False)
    Xs, ts, gs = Xtr.iloc[sub], np.asarray(types)[sub], np.asarray(groups)[sub]
    discrete = np.array([('=' in c) for c in Xs.columns])
    parts = [b for _, b in GroupKFold(n_groups).split(Xs, groups=gs)]
    keep, per_class = set(), {}
    for c in np.unique(ts):
        scores = []
        for b in parts:
            yc = (ts[b] == c).astype(int)
            if yc.sum() < 5: continue
            scores.append(mutual_info_classif(Xs.values[b], yc, discrete_features=discrete, random_state=seed))
        if not scores: continue
        mi = pd.Series(np.min(scores, axis=0), index=Xs.columns)
        par = mi.groupby([enc.parent(col) for col in mi.index]).max().sort_values(ascending=False)
        yv = par.values; x = np.arange(1, len(yv) + 1)
        if len(yv) < 3 or yv[0] <= 0: continue
        d = np.abs((yv[-1] - yv[0]) * x - (x[-1] - x[0]) * yv + x[-1] * yv[0] - yv[-1] * x[0])
        k = int(np.argmax(d)) + 1
        per_class[str(c)] = list(par.index[:k]); keep |= set(par.index[:k])
    cols = [col for col in Xtr.columns if enc.parent(col) in keep]
    return cols, per_class


def oof_threshold(p_oof, y):
    """Decision threshold that maximises MCC on out-of-fold (held-out-host) training predictions."""
    from sklearn.metrics import matthews_corrcoef
    grid = np.unique(np.quantile(p_oof, np.linspace(0.005, 0.995, 199)))
    best = max(grid, key=lambda t: matthews_corrcoef(y, (p_oof >= t).astype(int)))
    return float(best)


def apply_threshold(proba, t):
    """Monotone remapping so that the decision at 0.5 equals the decision at threshold t (ranking unchanged)."""
    p = proba[:, 1]
    q = np.where(p < t, 0.5 * p / max(t, 1e-9), 0.5 + 0.5 * (p - t) / max(1 - t, 1e-9))
    return np.c_[1 - q, q]


In [ ]:
%%writefile run.py
"""One experimental run: python run.py <protocol P1..P4> <task bin|multi> <seed> [--models a,b]"""
import sys, os, json, time, pickle
import numpy as np, pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import *
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

protocol, task, seed = sys.argv[1], sys.argv[2], int(sys.argv[3])
ALL = ['LR', 'RF', 'LGBM', 'XGB_all', 'XGB_MI', 'MLP', 'CNN', 'H_insample', 'H_stack_MI', 'H_stack']
V2 = ['XGB_thr', 'XGB_cs', 'CF_H', 'CF_H_thr', 'CF_H_cs', 'CF_v2']
models = sys.argv[5].split(',') if len(sys.argv) > 5 and sys.argv[4] == '--models' else ALL
OUT = os.path.join(os.environ.get("RESULTS_DIR", "/home/claude/exp/results"), f"{protocol}_{task}_{seed}.json")
os.makedirs(os.path.dirname(OUT), exist_ok=True)

df = data_for_protocol(protocol)
tr, te = split_protocol(df, protocol, seed)
if task == 'bin':
    y = df['label'].values.astype(int); K = 2
else:
    present = [c for c in CLASSES if c in set(df['type'])]
    y = df['type'].map({c: i for i, c in enumerate(present)}).values.astype(int); K = len(present)
feat = feature_cols(df)
enc = Encoder().fit(df.iloc[tr][feat])
Xtr = enc.transform(df.iloc[tr][feat]); Xte = enc.transform(df.iloc[te][feat])
ytr, yte = y[tr], y[te]
res = dict(classes=(present if task != 'bin' else ['normal', 'attack']), protocol=protocol, task=task, seed=seed, n=len(df), n_train=len(tr), n_test=len(te),
           test_class_counts=pd.Series(yte).value_counts().sort_index().to_dict(), n_encoded=Xtr.shape[1], models={})
if protocol == 'P4':
    res['test_attack_hosts'] = sorted(df.iloc[te].loc[df.iloc[te]['label'] == 1, 'src_ip'].unique().tolist())
    res['train_attack_hosts'] = sorted(df.iloc[tr].loc[df.iloc[tr]['label'] == 1, 'src_ip'].unique().tolist())

t0 = time.time()
sel, mi_rank, k = mi_elbow(Xtr, ytr, enc, seed=seed)
res['mi'] = dict(k_attributes=k, n_columns=len(sel), ranking=mi_rank.round(4).to_dict(), time=time.time() - t0)
slog = lambda a: np.sign(a) * np.log1p(np.abs(a))   # tames heavy-tailed byte/packet counts for neural nets
sc_all = StandardScaler().fit(slog(Xtr.values)); sc_mi = StandardScaler().fit(slog(Xtr[sel].values))
A_tr, A_te = sc_all.transform(slog(Xtr.values)), sc_all.transform(slog(Xte.values))
M_tr, M_te = sc_mi.transform(slog(Xtr[sel].values)), sc_mi.transform(slog(Xte[sel].values))
if os.environ.get('PERM_SEED'):
    perm = np.random.RandomState(int(os.environ['PERM_SEED'])).permutation(M_tr.shape[1])
    M_tr, M_te = M_tr[:, perm], M_te[:, perm]; res['cnn_input_permutation'] = perm.tolist()
spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1) if K == 2 else 1.0
res['scale_pos_weight'] = float(spw)


PRED = {}
def record(name, fit_time, proba, infer_time, size_bytes, extra=None):
    if os.environ.get('SAVE_PRED'): PRED[name] = proba.astype(np.float32)
    m = metrics(yte, proba, K)
    m.update(train_s=fit_time, infer_us_per_flow=1e6 * infer_time / len(yte), size_kb=size_bytes / 1024)
    if extra: m.update(extra)
    res['models'][name] = m
    print(f"{protocol} {task} s{seed} {name:11s} acc={m['acc']:.4f} f1m={m['f1_macro']:.4f} mcc={m['mcc']:.4f} t={fit_time:.0f}s", flush=True)
    json.dump(res, open(OUT, 'w'), indent=1)


def timed(fn):
    t = time.time(); out = fn(); return out, time.time() - t


def sz(obj):
    return len(pickle.dumps(obj))


def xgb_size(m):
    return len(m.get_booster().save_raw('json'))


if 'LR' in models:
    from sklearn.linear_model import LogisticRegression
    m, ft = timed(lambda: LogisticRegression(max_iter=300, n_jobs=2).fit(A_tr, ytr))
    p, it = timed(lambda: m.predict_proba(A_te)); record('LR', ft, p, it, sz(m))
if 'RF' in models:
    from sklearn.ensemble import RandomForestClassifier
    m, ft = timed(lambda: RandomForestClassifier(100, n_jobs=2, random_state=seed, min_samples_leaf=1).fit(Xtr.values, ytr))
    p, it = timed(lambda: m.predict_proba(Xte.values)); record('RF', ft, p, it, sz(m))
if 'LGBM' in models:
    from lightgbm import LGBMClassifier
    # multi-class: the shared configuration diverges; use a more conservative one (see manuscript Section 4.1)
    lkw = dict(learning_rate=0.1, num_leaves=63) if K == 2 else dict(learning_rate=0.05, num_leaves=31, min_child_samples=20, min_sum_hessian_in_leaf=1.0)
    m, ft = timed(lambda: LGBMClassifier(n_estimators=300, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                         n_jobs=2, random_state=seed, verbose=-1, **lkw).fit(Xtr.values, ytr))
    p, it = timed(lambda: m.predict_proba(Xte.values)); record('LGBM', ft, p, it, len(m.booster_.model_to_string()))
if 'XGB_all' in models:
    m, ft = timed(lambda: xgb_model(K, seed, spw).fit(Xtr.values, ytr))
    p, it = timed(lambda: m.predict_proba(Xte.values)); record('XGB_all', ft, p, it, xgb_size(m))
if 'XGB_MI' in models:
    m, ft = timed(lambda: xgb_model(K, seed, spw).fit(Xtr[sel].values, ytr))
    p, it = timed(lambda: m.predict_proba(Xte[sel].values)); record('XGB_MI', ft, p, it, xgb_size(m))
    try:
        import shap
        ex = shap.TreeExplainer(m); sub = np.random.RandomState(seed).choice(len(Xte), min(3000, len(Xte)), replace=False)
        sv = ex.shap_values(Xte[sel].values[sub])
        sv = np.abs(np.array(sv))
        sv = sv.mean(axis=(0, 2)) if sv.ndim == 3 else sv.mean(0)
        res['shap_XGB_MI'] = dict(zip(sel, sv.tolist()))
    except Exception as e:
        res['shap_err'] = str(e)
if 'MLP' in models:
    import tensorflow as tf
    from tensorflow.keras import layers
    tf.keras.utils.set_random_seed(seed)
    mlp = tf.keras.Sequential([layers.Input((M_tr.shape[1],)), layers.Dense(128, activation='relu'), layers.Dropout(0.3),
                               layers.Dense(64, activation='relu'), layers.Dense(1 if K == 2 else K, activation='sigmoid' if K == 2 else 'softmax')])
    mlp.compile('adam', 'binary_crossentropy' if K == 2 else 'sparse_categorical_crossentropy')
    cb = [tf.keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)]
    _, ft = timed(lambda: mlp.fit(M_tr, ytr, validation_split=0.1, epochs=25, batch_size=512, callbacks=cb, verbose=0))
    p, it = timed(lambda: mlp.predict(M_te, batch_size=4096, verbose=0))
    p = np.c_[1 - p[:, 0], p[:, 0]] if K == 2 else p
    record('MLP', ft, p, it, mlp.count_params() * 4)

need_cnn = any(x in models for x in ['CNN', 'H_insample', 'H_stack_MI', 'H_stack'])
if need_cnn:
    (cnn, hist), ft_cnn = timed(lambda: fit_cnn(M_tr, ytr, K, seed))
    res['cnn_history'] = {k: [float(v) for v in vals] for k, vals in hist.items() if k in ('loss', 'val_loss')}
    cnn_bytes = cnn.count_params() * 4
    p, it = timed(lambda: cnn_proba(cnn, M_te)); record('CNN', ft_cnn, p, it, cnn_bytes, dict(epochs=len(hist['loss'])))
    (Z_tr_in, Z_te), it_emb = timed(lambda: (cnn_embed(cnn, M_tr), cnn_embed(cnn, M_te)))
    it_emb_te = it_emb * len(M_te) / (len(M_tr) + len(M_te))
    if 'H_insample' in models:
        m, ft = timed(lambda: xgb_model(K, seed, spw).fit(Z_tr_in, ytr))
        p, it = timed(lambda: m.predict_proba(Z_te)); record('H_insample', ft + ft_cnn, p, it + it_emb_te, xgb_size(m) + cnn_bytes)
    if 'H_stack_MI' in models or 'H_stack' in models:
        # cross-fitted CNN posteriors: out-of-fold on train, fold-average on test (no label leakage into the meta-learner)
        P_oof = np.zeros((len(M_tr), K)); P_te = np.zeros((len(M_te), K)); t = time.time()
        for f, (a, b) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed).split(M_tr, ytr)):
            cf, _ = fit_cnn(M_tr[a], ytr[a], K, seed + 100 + f)
            P_oof[b] = cnn_proba(cf, M_tr[b]); P_te += cnn_proba(cf, M_te) / 3
        ft_oof = time.time() - t
        P_oof, P_te = P_oof[:, 1:], P_te[:, 1:]  # drop redundant column
        for name, Xa, Xb in [('H_stack_MI', Xtr[sel].values, Xte[sel].values), ('H_stack', Xtr.values, Xte.values)]:
            if name not in models: continue
            m, ft = timed(lambda: xgb_model(K, seed, spw).fit(np.c_[Xa, P_oof], ytr))
            p, it = timed(lambda: m.predict_proba(np.c_[Xb, P_te]))
            imp = m.get_booster().get_score(importance_type='gain'); tot = sum(imp.values())
            share = sum(v for k2, v in imp.items() if int(k2[1:]) >= Xa.shape[1]) / tot
            record(name, ft + ft_oof, p, it + it_emb_te * 3, xgb_size(m) + 3 * cnn_bytes, dict(gain_share_cnn=share))

# ---------------------------------------------------------------- v2: host-aware cross-fitted stacking + ablation
if any(x in models for x in V2):
    g_tr = df.iloc[tr]['src_ip'].astype(str).values
    folds, fold_kind = host_folds(ytr, g_tr, 3)
    res['v2'] = dict(fold_kind=fold_kind)
    t = time.time()
    sel2, per_class = mi_classwise_stable(Xtr, df.iloc[tr]['type'].values, g_tr, enc, seed=seed)
    res['v2'].update(cs_columns=len(sel2), cs_attributes=sorted({enc.parent(c) for c in sel2}), cs_time=time.time() - t)
    sc2 = StandardScaler().fit(slog(Xtr[sel2].values))
    C_tr, C_te = sc2.transform(slog(Xtr[sel2].values)), sc2.transform(slog(Xte[sel2].values))

    def meta_oof(Xa, extra=None):
        """host-grouped OOF probabilities of the XGBoost (meta-)learner on the training set (binary only)"""
        p = np.zeros(len(ytr))
        for a, b in folds:
            Za = Xa[a] if extra is None else np.c_[Xa[a], extra[a]]
            Zb = Xa[b] if extra is None else np.c_[Xa[b], extra[b]]
            p[b] = xgb_model(K, seed, spw).fit(Za, ytr[a]).predict_proba(Zb)[:, 1]
        return p

    if 'XGB_thr' in models and K == 2:
        t = time.time(); m = xgb_model(K, seed, spw).fit(Xtr.values, ytr); thr = oof_threshold(meta_oof(Xtr.values), ytr); ft = time.time() - t
        p, it = timed(lambda: m.predict_proba(Xte.values)); record('XGB_thr', ft, apply_threshold(p, thr), it, xgb_size(m), dict(threshold=thr))
    if 'XGB_cs' in models:
        m, ft = timed(lambda: xgb_model(K, seed, spw).fit(Xtr[sel2].values, ytr))
        p, it = timed(lambda: m.predict_proba(Xte[sel2].values)); record('XGB_cs', ft, p, it, xgb_size(m))

    def host_posteriors(Min, Mte, tag):
        P_oof = np.zeros((len(Min), K)); P_te = np.zeros((len(Mte), K)); t = time.time()
        for f, (a, b) in enumerate(folds):
            cf, _ = fit_cnn(Min[a], ytr[a], K, seed + 200 + f)
            P_oof[b] = cnn_proba(cf, Min[b]); P_te += cnn_proba(cf, Mte) / len(folds)
        return P_oof[:, 1:], P_te[:, 1:], time.time() - t

    def stack(name, Po, Pt, fto, thr=False):
        m, ft = timed(lambda: xgb_model(K, seed, spw).fit(np.c_[Xtr.values, Po], ytr))
        p, it = timed(lambda: m.predict_proba(np.c_[Xte.values, Pt]))
        imp = m.get_booster().get_score(importance_type='gain'); tot = sum(imp.values())
        share = sum(v for k2, v in imp.items() if int(k2[1:]) >= Xtr.shape[1]) / tot
        extra = dict(gain_share_cnn=share)
        if thr and K == 2:
            t = time.time(); th = oof_threshold(meta_oof(Xtr.values, Po), ytr); ft += time.time() - t
            p = apply_threshold(p, th); extra['threshold'] = th
        record(name, ft + fto, p, it, xgb_size(m) + 3 * cnn_bytes if 'cnn_bytes' in globals() else xgb_size(m), extra)

    if 'CF_H' in models or 'CF_H_thr' in models:
        Po, Pt, fto = host_posteriors(M_tr, M_te, 'mi')
        if 'CF_H' in models: stack('CF_H', Po, Pt, fto)
        if 'CF_H_thr' in models: stack('CF_H_thr', Po, Pt, fto, thr=True)
    if 'CF_H_cs' in models or 'CF_v2' in models:
        Po, Pt, fto = host_posteriors(C_tr, C_te, 'cs')
        if 'CF_H_cs' in models: stack('CF_H_cs', Po, Pt, fto)
        if 'CF_v2' in models: stack('CF_v2', Po, Pt, fto, thr=True)
res['done'] = True
json.dump(res, open(OUT, 'w'), indent=1)
if os.environ.get('SAVE_PRED'):
    np.savez_compressed(OUT.replace('.json', '_pred.npz'), y=yte, **PRED)


In [ ]:
%%writefile zeroday.py
"""Leave-one-attack-family-out (unseen attack) evaluation on the feature-space-deduplicated data.
python zeroday.py <family> <seed>"""
import sys, os, json, time
import numpy as np, pandas as pd
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from common import *
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold

fam, seed = sys.argv[1], int(sys.argv[2])
OUT = os.path.join(os.environ.get("RESULTS_DIR", "/home/claude/exp/results"), f"ZD_{fam}_{seed}.json")
os.makedirs(os.path.dirname(OUT), exist_ok=True)
df, _ = dedup_feature_space(load_raw())
_mn = int(os.environ.get('MIN_CLASS_N', '0'))
if _mn:
    _vc = df['type'].value_counts(); df = df[df['type'].isin(_vc[_vc >= _mn].index)].reset_index(drop=True)
idx = np.arange(len(df)); is_fam = (df['type'] == fam).values
normal = idx[df['type'].values == 'normal']; others = idx[(~is_fam) & (df['type'].values != 'normal')]
n_tr, n_te = train_test_split(normal, test_size=0.2, random_state=seed)
o_tr, _ = train_test_split(others, test_size=0.2, random_state=seed, stratify=df['type'].values[others])
tr = np.r_[n_tr, o_tr]; te = np.r_[n_te, idx[is_fam]]
y = df['label'].values.astype(int); feat = feature_cols(df)
enc = Encoder().fit(df.iloc[tr][feat]); Xtr = enc.transform(df.iloc[tr][feat]); Xte = enc.transform(df.iloc[te][feat])
ytr, yte = y[tr], y[te]
sel, _, k = mi_elbow(Xtr, ytr, enc, seed=seed)
slog = lambda a: np.sign(a) * np.log1p(np.abs(a))
sc = StandardScaler().fit(slog(Xtr[sel].values)); M_tr, M_te = sc.transform(slog(Xtr[sel].values)), sc.transform(slog(Xte[sel].values))
spw = (ytr == 0).sum() / (ytr == 1).sum()
res = dict(family=fam, seed=seed, n_fam=int(is_fam.sum()), n_test_normal=len(n_te), k=k, models={})


def rec(name, p):
    pred = p.argmax(1); a = yte == 1
    res['models'][name] = dict(detection_rate=float(pred[a].mean()), fpr=float(pred[~a].mean()),
                               auc=float(__import__('sklearn.metrics', fromlist=['x']).roc_auc_score(yte, p[:, 1])))
    print(fam, name, res['models'][name], flush=True)


from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
rec('RF', RandomForestClassifier(100, n_jobs=2, random_state=seed).fit(Xtr.values, ytr).predict_proba(Xte.values))
rec('LGBM', LGBMClassifier(n_estimators=300, num_leaves=63, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                           n_jobs=2, random_state=seed, verbose=-1).fit(Xtr.values, ytr).predict_proba(Xte.values))
rec('XGB_all', xgb_model(2, seed, spw).fit(Xtr.values, ytr).predict_proba(Xte.values))
rec('XGB_MI', xgb_model(2, seed, spw).fit(Xtr[sel].values, ytr).predict_proba(Xte[sel].values))
if os.environ.get('CS_ONLY'):
    sel2, _ = mi_classwise_stable(Xtr, df.iloc[tr]['type'].values, df.iloc[tr]['src_ip'].astype(str).values, enc, seed=seed)
    res['cs_attributes'] = sorted({enc.parent(c) for c in sel2})
    rec('XGB_cs', xgb_model(2, seed, spw).fit(Xtr[sel2].values, ytr).predict_proba(Xte[sel2].values))
    json.dump(res, open(OUT, 'w'), indent=1); sys.exit(0)
if os.environ.get('NO_CNN'):
    json.dump(res, open(OUT, 'w'), indent=1); sys.exit(0)
cnn, _ = fit_cnn(M_tr, ytr, 2, seed); rec('CNN', cnn_proba(cnn, M_te))
if os.environ.get('FAST'):
    json.dump(res, open(OUT, 'w'), indent=1); sys.exit(0)
P_oof = np.zeros(len(M_tr)); P_te = np.zeros(len(M_te))
for f, (a, b) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed).split(M_tr, ytr)):
    cf, _ = fit_cnn(M_tr[a], ytr[a], 2, seed + 100 + f); P_oof[b] = cnn_proba(cf, M_tr[b])[:, 1]; P_te += cnn_proba(cf, M_te)[:, 1] / 3
rec('H_stack', xgb_model(2, seed, spw).fit(np.c_[Xtr.values, P_oof], ytr).predict_proba(np.c_[Xte.values, P_te]))
json.dump(res, open(OUT, 'w'), indent=1)


In [ ]:
%%writefile aggregate.py
"""Aggregate results into tables (JSON/CSV) and publication figures."""
import os, json, glob
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, friedmanchisquare

R = os.environ.get("RESULTS_DIR", "/home/claude/exp/results")
FIG = os.environ.get("FIG_DIR", "/home/claude/paper/fig"); os.makedirs(FIG, exist_ok=True)
OUT = os.environ.get("AGG_OUT", "/home/claude/exp/agg.json")
ORDER = ['LR', 'RF', 'LGBM', 'XGB_all', 'XGB_MI', 'MLP', 'CNN', 'H_insample', 'H_stack_MI', 'H_stack']
NAMES = {'LR': 'LR', 'RF': 'RF', 'LGBM': 'LightGBM', 'XGB_all': 'XGBoost (all)', 'XGB_MI': 'XGBoost (MI)', 'MLP': 'MLP',
         'CNN': '1D-CNN (MI)', 'H_insample': 'CNN→XGB in-sample', 'H_stack_MI': 'CF-MI-CNN-XGB [X_MI‖P̂]',
         'H_stack': 'CF-MI-CNN-XGB [X‖P̂]'}
CLASSES = ['normal', 'backdoor', 'ddos', 'dos', 'injection', 'mitm', 'password', 'ransomware', 'scanning', 'xss']
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9})

runs = []
for f in sorted(glob.glob(f"{R}/P*_*.json")):
    r = json.load(open(f))
    if not r.get('done'): continue
    for m, v in r['models'].items():
        if m.endswith('_diverged'): continue
        runs.append(dict(protocol=r['protocol'], task=r['task'], seed=r['seed'], model=m, **{k: v[k] for k in v if not isinstance(v[k], list)},
                         per_class_f1=v.get('per_class_f1')))
D = pd.DataFrame(runs)
agg = {}


def ms(s):
    return f"{s.mean():.4f} ± {s.std(ddof=1) if len(s) > 1 else 0:.4f}"


# ---- binary summary per protocol
B = D[D.task == 'bin']
tab = {}
for p in sorted(B.protocol.unique()):
    sub = B[B.protocol == p]; tab[p] = {}
    for m in ORDER:
        s = sub[sub.model == m]
        if len(s) == 0: continue
        tab[p][m] = {k: dict(mean=float(s[k].mean()), sd=float(s[k].std(ddof=1)) if len(s) > 1 else 0.0, n=len(s))
                     for k in ['acc', 'mcc', 'f1_macro', 'precision', 'recall', 'fpr', 'auc', 'pr_auc', 'train_s', 'infer_us_per_flow', 'size_kb']}
agg['binary'] = tab

# pooled confusion for P4 (all folds)
pool = {}
for m in ORDER:
    s = B[(B.protocol == 'P4') & (B.model == m)]
    if len(s):
        tn, fp, fn, tp = [float(x) for x in (s.tn.sum(), s.fp.sum(), s.fn.sum(), s.tp.sum())]
        sdiv = lambda a, b: float(a / b) if b else float('nan')
        mcc = sdiv(tp * tn - fp * fn, np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)))
        pool[m] = dict(tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp), mcc=mcc, recall=sdiv(tp, tp + fn),
                       fpr=sdiv(fp, fp + tn), precision=sdiv(tp, tp + fp), folds=len(s))
agg['P4_pooled'] = pool

# ---- statistics: paired over all binary runs in P2-P4
S = B[B.protocol.isin(['P2', 'P3', 'P4'])].pivot_table(index=['protocol', 'seed'], columns='model', values='mcc').dropna()
stats = dict(n_pairs=len(S))
ref = 'H_stack'
for m in ORDER:
    if m == ref or m not in S: continue
    d = S[ref] - S[m]
    try:
        p = wilcoxon(S[ref], S[m]).pvalue if (d != 0).any() else 1.0
    except Exception:
        p = float('nan')
    stats[m] = dict(mean_diff=float(d.mean()), wins=int((d > 0).sum()), losses=int((d < 0).sum()), p=float(p))
ranks = S[[m for m in ORDER if m in S]].rank(axis=1, ascending=False).mean().sort_values()
stats['mean_rank'] = ranks.to_dict()
try:
    stats['friedman_p'] = float(friedmanchisquare(*[S[m] for m in ranks.index]).pvalue)
except Exception:
    pass
agg['stats_mcc'] = stats

# ---- MI / k / cnn gain share / shap
mi = {}
for f in sorted(glob.glob(f"{R}/P*_bin_*.json")):
    r = json.load(open(f))
    if not r.get('done') or 'mi' not in r: continue
    mi[f"{r['protocol']}_{r['seed']}"] = dict(k=r['mi']['k_attributes'], top=list(r['mi']['ranking'].items())[:r['mi']['k_attributes']],
                                              gain_share=r['models'].get('H_stack', {}).get('gain_share_cnn'),
                                              gain_share_mi=r['models'].get('H_stack_MI', {}).get('gain_share_cnn'),
                                              shap=r.get('shap_XGB_MI'), test_counts=r['test_class_counts'], spw=r['scale_pos_weight'])
    if r['protocol'] == 'P4': mi[f"P4_{r['seed']}"]['test_attack_hosts'] = r.get('test_attack_hosts')
agg['runs_meta'] = mi

# ---- multi-class
M = D[D.task == 'multi']
mt = {}
for p in sorted(M.protocol.unique()):
    sub = M[M.protocol == p]; mt[p] = {}
    for m in ORDER:
        s = sub[sub.model == m]
        if len(s) == 0: continue
        pcf = np.array([x for x in s.per_class_f1])
        mt[p][m] = dict(acc=float(s.acc.mean()), mcc=float(s.mcc.mean()), mcc_sd=float(s.mcc.std(ddof=1)) if len(s) > 1 else 0.0,
                        f1_macro=float(s.f1_macro.mean()), f1_macro_sd=float(s.f1_macro.std(ddof=1)) if len(s) > 1 else 0.0,
                        per_class_f1=pcf.mean(0).tolist(), n=len(s))
agg['multi'] = mt

# ---- zero-day
zd = {}
for f in sorted(glob.glob(f"{R}/ZD_*.json")):
    r = json.load(open(f)); zd[r['family']] = dict(n=r['n_fam'], **{m: v for m, v in r['models'].items()})
agg['zeroday'] = zd
json.dump(agg, open(OUT, 'w'), indent=1)

# ================= figures
cols = {'LR': '#9e9e9e', 'RF': '#8d6e63', 'LGBM': '#43a047', 'XGB_all': '#1e88e5', 'XGB_MI': '#90caf9', 'MLP': '#ffb74d',
        'CNN': '#fb8c00', 'H_insample': '#e53935', 'H_stack_MI': '#ba68c8', 'H_stack': '#6a1b9a'}
prots = [p for p in ['P1', 'P2', 'P3', 'P4'] if p in tab]
if prots:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    for ax, met, lab in [(axes[0], 'mcc', 'MCC'), (axes[1], 'f1_macro', 'Macro-F1')]:
        w = 0.08; x = np.arange(len(prots))
        for i, m in enumerate(ORDER):
            mu = [tab[p].get(m, {}).get(met, {}).get('mean', np.nan) for p in prots]
            sd = [tab[p].get(m, {}).get(met, {}).get('sd', 0) for p in prots]
            sd = [0 if p == 'P4' else v for p, v in zip(prots, sd)]
            ax.bar(x + (i - 4.5) * w, mu, w, yerr=sd, color=cols[m], label=NAMES[m], capsize=1.5, error_kw=dict(lw=0.6))
            if 'P4' in prots:
                pts = B[(B.protocol == 'P4') & (B.model == m)][met].values
                ax.scatter(np.full(len(pts), prots.index('P4') + (i - 4.5) * w), pts, s=4, color='k', zorder=3)
        ax.set_xticks(x); ax.set_xticklabels(prots); ax.set_ylabel(lab); ax.set_ylim(0.0, 1.005); ax.grid(axis='y', alpha=0.3)
    axes[1].legend(fontsize=7, ncol=1, bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
    plt.tight_layout(); plt.savefig(f"{FIG}/fig_protocol_mcc.png", dpi=300); plt.close()

if 'P4' in tab:
    s4 = B[B.protocol == 'P4']
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    for m in ['XGB_all', 'LGBM', 'RF', 'XGB_MI', 'CNN', 'H_insample', 'H_stack']:
        s = s4[s4.model == m].sort_values('seed')
        ax.plot(s.seed + 1, s.mcc, marker='o', label=NAMES[m], color=cols[m], lw=1.2)
    ax.set_xlabel('Source-host-disjoint fold'); ax.set_ylabel('MCC'); ax.set_xticks(range(1, 6)); ax.grid(alpha=0.3)
    ax.legend(fontsize=7, bbox_to_anchor=(1.01, 1), loc='upper left', frameon=False)
    plt.tight_layout(); plt.savefig(f"{FIG}/fig_p4_folds.png", dpi=300); plt.close()

if mt:
    p = 'P4' if 'P4' in mt else list(mt)[0]
    ms_ = [m for m in ['LR', 'RF', 'LGBM', 'XGB_all', 'XGB_MI', 'MLP', 'CNN', 'H_insample', 'H_stack'] if m in mt[p]]
    H = np.array([mt[p][m]['per_class_f1'] for m in ms_])
    fig, ax = plt.subplots(figsize=(8, 3.8))
    im = ax.imshow(H, cmap='viridis', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(10)); ax.set_xticklabels(CLASSES, rotation=35, ha='right'); ax.set_yticks(range(len(ms_))); ax.set_yticklabels([NAMES[m] for m in ms_])
    for i in range(H.shape[0]):
        for j in range(H.shape[1]):
            ax.text(j, i, f"{H[i, j]:.2f}", ha='center', va='center', fontsize=6.5, color='w' if H[i, j] < 0.6 else 'k')
    plt.colorbar(im, ax=ax, fraction=0.03); ax.set_title(f'Per-class F1, ten-class task ({p})', fontsize=9)
    plt.tight_layout(); plt.savefig(f"{FIG}/fig_multiclass_heatmap.png", dpi=300); plt.close()

if zd:
    fams = sorted(zd, key=lambda k: -zd[k].get('XGB_all', {}).get('detection_rate', 0))
    ms_ = ['RF', 'LGBM', 'XGB_all', 'XGB_MI', 'CNN', 'H_stack']
    fig, ax = plt.subplots(figsize=(9, 3.4)); w = 0.13; x = np.arange(len(fams))
    for i, m in enumerate(ms_):
        ax.bar(x + (i - 2.5) * w, [zd[f][m]['detection_rate'] for f in fams], w, color=cols[m], label=NAMES[m])
    ax.set_xticks(x); ax.set_xticklabels(fams, rotation=20); ax.set_ylabel('Detection rate of unseen family'); ax.set_ylim(0, 1.02)
    ax.legend(fontsize=7, ncol=3, frameon=False); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{FIG}/fig_zeroday.png", dpi=300); plt.close()

# MI ranking figure (P3 seed 0) and SHAP
k0 = 'P3_0' if 'P3_0' in mi else (list(mi)[0] if mi else None)
if k0:
    r = json.load(open(f"{R}/{k0.split('_')[0]}_bin_{k0.split('_')[1]}.json"))
    rk = pd.Series(r['mi']['ranking']); k = r['mi']['k_attributes']
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ax = axes[0]; ax.plot(range(1, len(rk) + 1), rk.values, 'o-', ms=3, color='#1e88e5')
    ax.axvline(k, ls='--', color='#e53935'); ax.text(k + 0.5, rk.max() * 0.9, f'knee: k = {k}', color='#e53935', fontsize=8)
    for i in range(min(k, 12)):
        ax.text(i + 1.3, rk.values[i], rk.index[i], fontsize=6.5, rotation=0, va='center')
    ax.set_xlabel('Attribute rank'); ax.set_ylabel('MI with label (nats, max over encoded columns)'); ax.grid(alpha=0.3)
    sh = pd.Series(r.get('shap_XGB_MI', {}))
    if len(sh):
        sh = sh.sort_values().tail(15); ax = axes[1]
        ax.barh(range(len(sh)), sh.values, color='#6a1b9a'); ax.set_yticks(range(len(sh))); ax.set_yticklabels(sh.index, fontsize=7)
        ax.set_xlabel('mean |SHAP| (XGBoost on MI-selected features)'); ax.grid(axis='x', alpha=0.3)
    plt.tight_layout(); plt.savefig(f"{FIG}/fig_mi_shap.png", dpi=300); plt.close()

# CNN learning curve
if os.path.exists(f"{R}/P3_bin_0.json"):
    r = json.load(open(f"{R}/P3_bin_0.json")); h = r.get('cnn_history')
    if h:
        fig, ax = plt.subplots(figsize=(5, 3))
        ax.plot(range(1, len(h['loss']) + 1), h['loss'], label='train'); ax.plot(range(1, len(h['val_loss']) + 1), h['val_loss'], label='validation')
        ax.set_xlabel('Epoch'); ax.set_ylabel('Binary cross-entropy'); ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout(); plt.savefig(f"{FIG}/fig_cnn_curve.png", dpi=300); plt.close()
print('aggregated', len(D), 'model-runs')


In [ ]:
%%writefile p4_host_folds.json
{"192.168.1.190":0,"192.168.1.34":0,"192.168.1.36":0,"192.168.1.79":1,"192.168.1.133":1,"192.168.1.103":1,"192.168.1.194":1,"192.168.1.17":1,"192.168.1.30":1,"fe80::a691:b1ff:fe1e:5790":1,"fe80::8520:4eba:6770:702a":1,"fe80::855:e4ef:de26:7aca":1,"fe80::f554:8774:9968:b60b":1,"192.168.1.6":1,"192.168.1.184":1,"172.17.0.3":1,"192.168.1.195":2,"192.168.1.39":2,"192.168.1.38":2,"192.168.1.33":2,"192.168.1.37":2,"192.168.1.152":3,"192.168.1.250":3,"192.168.1.192":3,"192.168.1.46":3,"192.168.1.1":3,"2405:6e00:10ce:2c00:20c:29ff:feee:e07a":3,"2405:6e00:10ce:2c00:9064:17:85d2:57d4":3,"fe80::20c:29ff:feee:e07a":3,"2405:6e00:10ce:2c00:a691:b1ff:fe1e:5790":3,"0.0.0.0":3,"fe80::1087:c341:7263:ead3":3,"fe80::601f:9cde:4832:2f3f":3,"fe80::343b:2cf:e9fb:6570":3,"fe80::20c:29ff:fed2:b002":3,"fe80::76:ad3e:1329:a7fd":3,"fe80::e4a6:b91f:48f3:ac01":3,"2405:6e00:10ce:2c00:3924:f1bc:16a9:e16c":3,"2405:6e00:10ce:2c00:8520:4eba:6770:702a":3,"192.168.1.31":3,"192.168.1.169":3,"172.17.0.7":3,"192.168.1.49":3,"3.122.49.24":4,"192.168.1.193":4,"192.168.1.191":4,"127.0.0.1":4,"fe80::2c3:f4ff:fe0f:6773":4,"2405:6e00:10ce:2c00:c058:dbaa:f212:a2d8":4,"2405:6e00:10ce:2c00:20c:29ff:fed2:b002":4,"2600:c0d:0:1::932":4,"2600:c0d:0:1::928":4,"2600:c0d:0:1::940":4,"2600:c0d:0:1::936":4,"2600:c0d:0:1::916":4,"fe80::ee1f:72ff:fef1:286d":4,"2600:3c01::f03c:91ff:fe31:4f8c":4,"2a01:7e00::f03c:91ff:feb2:e8c1":4,"2400:8901::f03c:91ff:feb2:455c":4,"2405:6e00:10ce:2c00:2cf2:ff40:a9bf:1fac":4,"2405:6e00:10ce:2c00:7055:d341:454e:4328":4,"2405:6e00:10ce:2c00:39d0:b970:40d3:995b":4,"192.168.1.32":4,"192.168.1.180":4,"251.117.144.201":4,"179.138.33.201":4,"102.126.207.233":4,"61.186.10.138":4,"165.93.175.246":4,"33.72.20.181":4,"207.201.99.161":4,"82.252.43.4":4,"133.250.32.136":4,"136.133.241.49":4,"241.54.88.140":4,"76.123.209.109":4,"3.244.143.54":4,"142.162.90.3":4,"198.156.5.210":4,"2.66.109.216":4,"102.145.79.7":4,"27.62.172.118":4,"38.195.64.217":4,"117.193.156.39":4,"49.232.208.178":4,"117.208.253.121":4,"108.197.203.131":4,"186.162.11.14":4,"185.8.174.9":4,"252.219.24.216":4,"22.82.99.62":4,"78.48.178.75":4,"131.23.100.241":4,"157.2.235.93":4,"83.197.162.146":4,"8.57.213.34":4,"116.114.12.135":4,"197.32.212.14":4,"40.54.177.172":4,"128.215.246.89":4,"206.115.68.88":4,"214.127.210.1":4,"253.219.234.52":4,"45.12.239.180":4,"153.227.247.161":4,"83.93.241.188":4,"195.42.7.185":4,"162.2.96.32":4,"173.154.157.34":4,"97.128.54.254":4,"82.57.134.9":4,"246.209.245.59":4,"163.11.223.139":4,"43.200.20.13":4,"31.204.194.133":4,"38.245.16.46":4,"74.160.220.99":4,"70.167.212.197":4,"113.87.159.74":4,"173.234.182.246":4,"126.112.159.253":4,"195.244.30.23":4,"121.182.237.13":4,"145.3.254.29":4,"17.173.156.192":4,"29.231.141.142":4,"64.102.244.19":4,"175.74.87.242":4,"241.145.29.40":4,"80.2.219.204":4,"96.30.246.80":4,"197.32.119.151":4,"17.86.104.109":4,"24.179.80.28":4,"100.254.136.190":4,"19.228.126.230":4,"152.211.229.69":4,"131.255.190.97":4,"204.14.181.120":4,"84.182.199.37":4,"252.162.78.63":4,"90.242.240.141":4,"189.184.145.223":4,"59.135.167.67":4,"13.205.87.191":4,"120.245.207.213":4,"250.135.85.46":4,"46.221.67.193":4,"165.140.25.64":4,"38.32.24.71":4,"15.175.14.175":4,"135.213.87.124":4,"89.208.140.2":4,"169.170.164.8":4,"43.165.128.160":4,"170.170.58.189":4,"27.119.80.152":4,"58.227.237.120":4,"69.214.38.44":4,"166.204.187.165":4,"36.45.171.223":4,"39.167.60.172":4,"129.68.225.96":4,"2.240.1.243":4,"188.140.73.167":4,"80.99.105.37":4,"104.131.227.159":4,"80.233.23.92":4,"103.91.6.9":4,"109.192.191.144":4,"210.60.56.221":4,"24.123.202.194":4,"86.144.204.133":4,"80.192.225.34":4,"4.3.204.213":4,"174.186.201.232":4,"142.153.105.13":4,"4.198.229.173":4,"123.31.68.148":4,"41.9.47.169":4,"173.141.164.151":4,"53.32.66.142":4,"83.47.207.0":4,"146.172.40.157":4,"62.252.143.197":4,"162.0.147.197":4,"150.45.33.151":4,"59.131.155.58":4,"122.200.94.196":4,"130.27.133.98":4,"126.75.121.247":4,"173.246.39.188":4,"181.202.164.243":4,"253.135.251.231":4,"96.242.255.134":4,"69.43.150.130":4,"204.4.171.176":4,"7.148.147.151":4,"244.115.251.190":4,"212.11.205.100":4,"61.28.241.255":4,"76.48.242.14":4,"146.49.176.147":4,"134.126.191.176":4,"208.24.155.204":4,"214.60.75.125":4,"2.252.129.9":4,"248.55.74.165":4,"203.56.13.144":4,"161.99.15.93":4,"169.58.166.217":4,"13.207.55.39":4,"135.182.133.16":4,"99.199.64.58":4,"8.115.178.200":4,"107.15.210.1":4,"253.172.118.221":4,"63.235.126.30":4,"82.92.48.238":4,"1.85.207.205":4,"48.19.228.79":4,"37.198.148.61":4,"135.228.231.247":4,"220.60.243.195":4,"183.244.77.236":4,"99.0.121.108":4,"31.191.59.181":4,"76.106.236.114":4,"141.195.55.27":4,"7.186.109.35":4,"109.156.249.183":4,"138.42.153.195":4,"249.250.243.77":4,"197.84.121.239":4,"191.66.145.19":4,"13.100.85.152":4,"45.112.116.242":4,"85.124.150.247":4,"118.54.133.152":4,"102.224.13.34":4,"94.125.154.168":4,"181.184.222.34":4,"22.95.155.5":4,"193.79.242.135":4,"173.96.103.44":4,"176.140.26.184":4,"184.136.130.27":4,"38.83.138.82":4,"204.36.146.178":4,"156.227.85.94":4,"167.38.133.64":4,"198.194.36.231":4,"196.76.74.179":4,"253.56.163.156":4,"20.85.243.192":4,"109.77.162.65":4,"171.54.145.157":4,"69.180.171.147":4,"2.95.201.142":4,"7.66.89.102":4,"33.161.20.144":4,"156.23.28.28":4,"85.99.223.28":4,"48.227.199.85":4,"56.232.101.217":4,"129.130.253.242":4,"186.19.50.145":4,"38.55.26.144":4,"88.110.153.227":4,"72.213.33.104":4,"5.253.188.164":4,"202.185.0.111":4,"44.134.116.198":4,"59.255.235.65":4,"5.31.210.75":4,"90.174.41.148":4,"69.16.156.119":4,"133.229.227.194":4,"60.129.154.254":4,"60.206.137.216":4,"155.90.153.244":4,"209.142.230.224":4,"87.104.237.170":4,"109.23.24.180":4,"147.160.210.51":4,"149.176.220.142":4,"169.49.125.118":4,"58.194.234.215":4,"247.186.114.252":4,"51.48.134.8":4,"188.56.209.43":4,"21.228.153.30":4,"155.74.117.214":4,"42.169.25.171":4,"128.94.170.207":4,"82.197.84.121":4,"192.12.192.242":4,"192.0.32.59":4,"139.135.7.174":4,"223.165.71.130":4,"156.7.107.23":4,"211.202.250.207":4,"61.239.72.27":4,"41.180.142.252":4,"98.255.18.218":4,"113.29.52.74":4,"187.156.53.231":4,"41.64.140.170":4,"17.202.175.229":4,"93.13.236.88":4,"163.170.190.9":4,"170.119.79.129":4,"75.37.174.35":4,"252.133.219.183":4,"242.70.12.109":4,"217.24.116.127":4,"243.151.97.221":4,"44.164.104.161":4,"68.229.203.61":4,"242.203.22.98":4,"181.59.67.133":4,"41.159.171.146":4,"21.22.13.5":4,"206.51.149.234":4,"203.113.131.2":4,"245.76.88.163":4,"188.238.112.170":4,"241.226.197.111":4,"128.174.13.147":4,"38.178.19.100":4,"193.179.254.181":4,"145.250.5.80":4,"78.195.103.162":4,"126.48.202.103":4,"100.201.169.244":4,"245.172.51.27":4,"191.6.122.200":4,"27.184.187.78":4,"76.250.212.123":4,"188.254.80.24":4,"19.29.10.162":4,"185.39.208.67":4,"107.214.233.21":4,"202.31.57.180":4,"166.161.35.45":4,"133.137.144.16":4,"211.177.6.28":4,"88.199.230.214":4,"209.106.166.96":4,"166.190.160.98":4,"204.113.200.113":4,"123.85.239.215":4,"240.37.122.202":4,"179.43.106.165":4,"86.162.156.175":4,"16.22.33.175":4,"179.191.190.19":4,"117.193.160.202":4,"39.119.86.251":4,"137.106.202.22":4,"47.180.11.83":4,"144.153.30.66":4,"153.164.59.146":4,"142.210.117.81":4,"135.181.109.74":4,"206.193.123.60":4,"86.31.224.31":4,"49.64.94.211":4,"83.150.222.83":4,"141.74.40.212":4,"104.168.248.112":4,"88.136.121.203":4,"67.18.209.201":4,"25.244.98.111":4,"166.197.161.9":4,"93.198.62.191":4,"21.246.50.230":4,"153.28.147.12":4,"69.168.248.131":4,"123.118.10.50":4,"70.64.45.69":4,"98.53.61.209":4,"111.68.47.49":4,"120.147.194.71":4,"1.36.186.16":4,"210.63.112.130":4,"56.124.235.25":4,"201.17.185.133":4,"183.193.82.223":4,"143.7.74.33":4,"44.141.193.79":4,"60.13.1.60":4,"144.165.37.249":4,"2.251.116.227":4,"129.119.31.166":4,"212.92.55.248":4,"203.254.251.24":4,"82.231.114.150":4,"37.220.93.87":4,"130.57.7.120":4,"182.124.195.153":4,"185.202.44.45":4,"4.33.219.112":4,"158.26.244.231":4,"176.13.167.209":4,"28.205.202.125":4,"179.146.10.213":4,"243.47.195.204":4,"87.17.22.194":4,"64.124.75.47":4,"6.24.13.169":4,"113.171.46.135":4,"123.88.139.119":4,"215.122.117.75":4,"208.212.34.78":4,"167.32.122.97":4,"27.44.151.39":4,"65.95.94.247":4,"34.165.207.244":4,"144.230.52.228":4,"126.119.123.10":4,"155.46.138.188":4,"174.99.103.26":4,"63.97.227.107":4,"94.48.255.73":4,"193.251.221.252":4,"168.134.78.66":4,"85.98.170.220":4,"142.182.152.51":4,"73.159.153.140":4,"194.72.56.153":4,"137.135.90.192":4,"178.72.214.96":4,"83.143.19.137":4,"69.79.13.34":4,"99.93.10.11":4,"211.10.74.181":4,"165.73.147.69":4,"151.128.75.49":4,"64.90.49.28":4,"57.182.246.50":4,"11.4.84.102":4,"176.226.25.211":4,"198.30.182.177":4,"112.72.85.49":4,"4.19.156.238":4,"84.14.76.59":4,"17.138.43.55":4,"94.20.157.139":4,"80.30.9.241":4,"31.140.224.9":4,"111.175.220.46":4,"196.150.167.23":4,"77.16.132.214":4,"121.241.106.175":4,"202.218.192.241":4,"40.188.97.224":4,"61.241.147.117":4,"43.20.223.129":4,"206.68.70.146":4,"103.245.59.202":4,"180.7.252.201":4,"188.120.1.99":4,"60.234.4.124":4,"141.161.176.29":4,"5.114.114.133":4,"205.114.168.195":4,"3.181.79.64":4,"41.237.74.231":4,"223.45.233.48":4,"131.218.185.117":4,"182.218.113.217":4,"223.238.46.68":4,"42.151.56.163":4,"196.56.59.95":4,"37.230.49.202":4,"175.46.126.88":4,"31.46.171.94":4,"41.56.44.19":4,"47.65.255.230":4,"164.104.117.14":4,"106.2.226.3":4,"210.144.205.110":4,"15.120.210.145":4,"150.134.73.232":4,"92.114.5.106":4,"135.171.38.197":4,"202.204.135.227":4,"66.115.219.177":4,"111.242.251.92":4,"88.72.112.207":4,"124.224.119.118":4,"130.179.150.148":4,"36.142.180.131":4,"48.226.219.0":4,"210.179.241.143":4,"44.173.38.27":4,"161.75.159.169":4,"151.193.152.230":4,"173.92.244.56":4,"81.122.223.109":4,"99.154.5.230":4,"190.226.126.59":4,"220.73.186.115":4,"54.168.66.215":4,"134.153.81.237":4,"68.229.156.64":4,"57.108.168.204":4,"96.161.188.182":4,"88.73.205.176":4,"114.24.247.150":4,"113.11.63.155":4,"62.28.190.166":4,"177.92.166.69":4,"95.234.126.129":4,"3.76.54.231":4,"145.235.45.237":4,"12.19.188.73":4,"170.198.75.193":4,"198.1.94.24":4,"119.181.114.2":4,"142.69.253.245":4,"137.98.44.10":4,"181.57.253.82":4,"159.218.220.174":4,"59.172.228.222":4,"6.26.39.186":4,"209.152.7.2":4,"184.196.87.240":4,"154.93.181.165":4,"156.70.220.5":4,"125.163.42.222":4,"21.54.70.171":4,"205.44.138.111":4,"174.144.12.165":4,"111.204.247.151":4,"52.253.160.33":4,"73.178.247.136":4,"36.119.171.16":4,"46.225.51.131":4,"126.234.119.173":4,"33.31.55.82":4,"131.119.145.24":4,"70.251.6.98":4,"16.157.79.229":4,"103.121.145.3":4,"55.90.45.97":4,"4.135.184.108":4,"90.72.162.93":4,"197.102.253.21":4,"59.67.118.74":4,"200.65.47.39":4,"9.135.207.155":4,"71.224.117.147":4,"199.248.98.101":4,"210.129.37.183":4,"61.83.91.36":4,"99.254.92.218":4,"62.245.24.234":4,"165.34.124.34":4,"135.42.78.20":4,"165.51.208.241":4,"203.241.79.28":4,"33.82.154.131":4,"1.47.238.165":4,"41.123.9.99":4,"67.80.73.2":4,"192.96.221.255":4,"134.239.87.242":4,"152.149.170.44":4,"11.173.51.57":4,"156.237.123.83":4,"11.59.92.199":4,"192.223.107.135":4,"75.138.27.205":4,"46.137.128.52":4,"177.125.116.253":4,"10.58.13.144":4,"90.33.57.163":4,"23.83.98.254":4,"145.226.169.147":4,"49.237.119.195":4,"115.104.179.121":4,"27.252.57.118":4,"16.246.14.180":4,"181.103.216.138":4,"56.229.98.228":4,"183.34.231.237":4,"34.181.123.54":4,"117.164.84.26":4,"49.42.28.224":4,"97.86.192.166":4,"27.150.244.184":4,"171.101.206.67":4,"55.193.207.123":4,"197.98.162.200":4,"105.197.58.249":4,"19.26.124.180":4,"41.239.151.45":4,"89.242.67.180":4,"146.33.226.58":4,"214.38.64.182":4,"220.70.175.222":4,"137.77.83.240":4,"18.48.32.172":4,"41.253.251.99":4,"125.57.173.133":4,"137.241.15.151":4,"140.196.11.83":4,"184.4.78.210":4,"168.31.11.141":4,"59.61.198.54":4,"164.186.30.249":4,"174.92.189.12":4,"52.26.116.85":4,"58.17.220.106":4,"187.16.245.127":4,"104.196.21.170":4,"5.230.236.255":4,"67.81.47.31":4,"86.247.149.218":4,"3.240.116.118":4,"123.209.110.108":4,"80.255.128.163":4,"46.11.171.124":4,"95.18.247.12":4,"99.149.67.134":4,"157.193.200.22":4,"55.110.153.52":4,"210.189.17.210":4,"204.138.49.247":4,"202.65.159.178":4,"204.162.150.60":4,"20.75.238.0":4,"83.241.67.226":4,"12.187.1.164":4,"12.221.229.4":4,"9.79.187.61":4,"32.156.117.100":4,"208.109.195.180":4,"40.85.148.153":4,"178.55.41.123":4,"177.186.161.253":4,"73.51.119.150":4,"13.187.135.177":4,"156.112.114.136":4,"173.239.53.230":4,"112.59.228.208":4,"36.121.114.190":4,"186.249.188.67":4,"117.156.12.71":4,"108.60.191.159":4,"69.1.58.88":4,"2.179.77.210":4,"130.62.48.160":4,"65.162.250.186":4,"28.58.148.80":4,"126.155.143.67":4,"118.209.106.56":4,"200.129.147.65":4,"48.155.88.115":4,"26.86.88.136":4,"101.196.143.32":4,"56.139.243.229":4,"58.157.27.249":4,"109.210.11.254":4,"138.30.200.93":4,"113.193.166.170":4,"114.93.188.231":4,"191.72.213.238":4,"207.225.67.135":4,"83.130.110.254":4,"14.89.198.49":4,"135.155.154.202":4,"29.192.182.146":4,"10.82.20.186":4,"42.41.120.65":4,"169.101.92.132":4,"19.20.153.205":4,"104.146.164.116":4,"151.246.152.35":4,"44.88.247.135":4,"128.184.203.90":4,"105.182.237.223":4,"103.86.45.106":4,"166.242.191.27":4,"27.104.83.200":4,"81.143.246.197":4,"193.169.126.41":4,"10.173.8.137":4,"191.217.162.74":4,"113.208.151.68":4,"193.219.111.99":4,"96.106.242.13":4,"76.246.233.64":4,"153.2.27.143":4,"124.95.36.157":4,"1.209.92.29":4,"26.156.156.143":4,"22.146.99.56":4,"35.116.189.164":4,"158.79.236.121":4,"212.84.185.22":4,"219.255.183.230":4,"5.4.131.142":4,"110.52.125.163":4,"152.129.8.56":4,"145.151.255.218":4,"158.90.129.24":4,"61.162.179.38":4,"169.250.31.214":4,"152.72.84.160":4,"67.142.114.182":4,"121.190.20.144":4,"159.47.148.104":4,"94.40.28.249":4,"143.25.64.56":4,"131.144.24.45":4,"35.2.104.48":4,"129.163.131.43":4,"69.164.21.150":4,"195.238.248.106":4,"86.48.123.189":4,"101.85.155.118":4,"168.44.194.142":4,"34.57.59.238":4,"113.150.129.154":4,"62.128.105.163":4,"36.181.103.194":4,"191.87.207.101":4,"206.211.175.213":4,"179.45.125.23":4,"26.253.163.170":4,"136.227.31.237":4,"128.57.47.34":4,"58.43.91.0":4,"204.212.253.190":4,"194.163.26.34":4,"88.125.253.192":4,"173.243.9.18":4,"87.57.169.249":4,"113.67.217.34":4,"131.141.184.37":4,"129.38.249.195":4,"101.153.131.23":4,"79.82.172.207":4,"215.90.107.79":4,"147.27.253.96":4,"213.42.161.193":4,"190.211.106.120":4,"166.200.184.7":4,"54.138.215.36":4,"28.82.12.53":4,"113.142.255.94":4,"131.33.186.152":4,"154.73.64.126":4,"211.25.171.192":4,"98.95.26.77":4,"49.146.145.221":4,"123.27.160.55":4,"144.238.207.251":4,"143.157.110.134":4,"145.31.175.16":4,"77.239.162.21":4,"112.68.194.68":4,"34.53.193.148":4,"46.71.202.146":4,"154.194.214.228":4,"178.73.33.163":4,"138.107.88.151":4,"139.153.217.128":4,"22.187.218.96":4,"174.230.81.31":4,"137.106.174.221":4,"178.172.195.158":4,"57.249.66.38":4,"186.217.37.10":4,"134.6.72.47":4,"189.16.248.149":4,"58.96.208.152":4,"108.225.113.65":4,"18.29.24.0":4,"74.96.134.161":4,"123.95.68.67":4,"56.180.141.199":4,"18.163.72.236":4,"42.11.112.188":4,"49.44.181.164":4,"16.153.250.0":4,"20.104.83.99":4,"37.3.74.23":4,"83.92.141.226":4,"198.26.54.5":4,"203.21.3.43":4,"218.212.130.48":4,"168.93.62.104":4,"67.177.171.123":4,"44.202.37.236":4,"173.141.87.89":4,"20.58.149.112":4,"56.145.64.97":4,"139.139.143.109":4,"72.172.73.70":4,"10.252.98.126":4,"109.85.221.66":4,"105.48.61.153":4,"12.225.177.79":4,"157.59.248.8":4,"178.32.175.30":4,"81.194.200.17":4,"166.116.5.252":4,"222.193.190.104":4,"221.46.222.148":4,"66.214.68.13":4,"152.27.211.190":4,"150.130.1.137":4,"132.176.47.37":4,"100.19.54.118":4,"108.114.65.16":4,"147.207.66.8":4,"136.20.179.187":4,"107.118.147.161":4,"190.34.243.205":4,"116.155.14.226":4,"41.75.182.250":4,"68.148.58.92":4,"163.101.42.234":4,"128.58.51.193":4,"110.24.178.186":4,"93.180.133.167":4,"186.96.114.160":4,"160.160.151.13":4,"195.84.2.168":4,"216.222.105.126":4,"170.54.230.239":4,"25.4.15.204":4,"101.104.23.175":4,"173.123.248.232":4,"5.191.245.42":4,"38.46.31.166":4,"42.244.138.62":4,"165.41.43.245":4,"159.29.40.94":4,"78.153.164.24":4,"200.67.175.95":4,"2.25.124.187":4,"203.130.100.57":4,"81.13.1.64":4,"168.38.85.151":4,"169.140.24.68":4,"184.9.131.212":4,"74.131.228.141":4,"134.163.255.143":4,"15.225.172.56":4,"134.207.151.117":4,"52.60.183.122":4,"90.89.249.77":4,"219.170.234.141":4,"5.218.57.254":4,"53.41.157.63":4,"195.155.201.238":4,"176.133.218.143":4,"107.18.227.61":4,"110.220.186.192":4,"89.128.228.13":4,"122.230.212.182":4,"14.149.248.99":4,"85.73.253.124":4,"217.121.204.70":4,"17.143.103.152":4,"150.100.229.100":4,"39.151.64.184":4,"162.36.91.156":4,"49.170.51.212":4,"156.62.225.45":4,"107.67.200.245":4,"110.244.251.121":4,"197.79.113.19":4,"141.179.173.108":4,"72.56.245.241":4,"124.68.132.178":4,"86.69.254.164":4,"202.62.73.47":4,"173.185.246.116":4,"49.215.97.142":4,"23.28.243.64":4,"6.105.80.54":4,"26.227.33.214":4,"24.118.47.49":4,"14.224.196.11":4,"175.91.80.97":4,"209.30.130.245":4,"187.127.240.246":4,"194.93.250.106":4,"178.9.70.73":4,"80.65.214.96":4,"17.24.188.221":4,"44.105.212.0":4,"205.33.143.220":4,"221.27.8.244":4,"96.173.52.135":4,"194.54.227.57":4,"139.134.144.243":4,"182.141.25.202":4,"86.151.6.27":4,"194.42.238.179":4,"153.15.131.12":4,"161.56.206.182":4,"197.94.212.186":4,"102.145.128.189":4,"71.179.7.32":4,"29.224.191.17":4,"86.234.136.178":4,"203.6.203.23":4,"12.167.65.31":4,"153.28.136.101":4,"148.161.183.87":4,"66.253.252.241":4,"22.30.148.179":4,"42.3.225.108":4,"160.170.29.107":4,"39.208.108.139":4,"160.61.106.194":4,"6.217.182.53":4,"96.57.188.168":4,"171.241.21.191":4,"15.181.39.160":4,"199.141.221.13":4,"109.220.96.66":4,"79.192.83.147":4,"24.21.201.174":4,"133.132.216.175":4,"180.90.238.145":4,"77.195.160.148":4,"72.152.85.48":4,"167.212.169.135":4,"215.5.196.252":4,"94.4.86.215":4,"38.253.62.69":4,"182.10.121.62":4,"62.33.214.58":4,"216.209.227.103":4,"54.156.110.67":4,"162.177.109.225":4,"59.22.244.109":4,"181.241.233.2":4,"82.71.143.144":4,"66.18.230.50":4,"206.109.241.80":4,"155.232.126.3":4,"193.235.117.195":4,"152.95.220.19":4,"65.117.71.31":4,"84.153.99.52":4,"204.19.29.111":4,"150.173.137.80":4,"99.86.56.3":4,"54.13.86.91":4,"55.117.251.44":4,"84.60.106.170":4,"1.240.202.245":4,"86.17.129.83":4,"146.202.108.91":4,"47.82.35.169":4,"119.159.203.152":4,"43.136.197.0":4,"56.233.101.31":4,"104.208.26.211":4,"152.161.79.225":4,"155.205.221.144":4,"77.245.211.33":4,"162.218.86.45":4,"148.224.241.170":4,"175.240.196.128":4,"75.242.90.151":4,"39.201.14.111":4,"29.195.84.194":4,"192.171.226.134":4,"136.200.248.237":4,"159.138.154.166":4,"202.118.188.35":4,"80.125.147.146":4,"199.5.225.217":4,"62.87.238.236":4,"26.158.173.167":4,"103.254.36.128":4,"14.158.160.27":4,"151.149.137.185":4,"119.199.150.108":4,"118.99.177.77":4,"101.117.165.128":4,"193.228.108.52":4,"210.209.9.1":4,"191.227.163.199":4,"51.89.135.124":4,"223.167.157.52":4,"1.222.181.213":4,"101.221.200.39":4,"51.211.116.83":4,"40.230.124.140":4,"117.139.123.68":4,"55.44.108.119":4,"215.58.149.192":4,"42.59.168.112":4,"152.163.10.167":4,"72.82.16.211":4,"145.165.76.8":4,"20.188.120.250":4,"82.158.150.215":4,"102.24.171.141":4,"54.17.65.78":4,"162.196.126.120":4,"139.80.131.41":4,"117.141.88.17":4,"28.97.248.16":4,"15.62.181.238":4,"3.234.119.109":4,"4.137.36.161":4,"36.177.164.187":4,"109.10.192.46":4,"124.77.2.156":4,"150.36.210.220":4,"148.93.252.10":4,"70.222.235.139":4,"106.194.52.49":4,"36.112.225.134":4,"11.112.78.33":4,"98.15.170.132":4,"168.195.195.101":4,"146.26.55.32":4,"167.199.137.43":4,"43.17.76.64":4,"211.139.179.249":4,"18.192.39.136":4,"202.96.75.72":4,"103.13.80.89":4,"159.134.237.59":4,"131.130.199.90":4,"172.17.0.5":4,"172.17.0.6":4,"252.130.40.41":4,"160.168.175.213":4,"70.62.150.17":4,"63.243.139.114":4,"70.51.85.183":4,"132.10.141.237":4,"31.37.51.194":4,"195.215.119.138":4,"12.77.123.210":4,"133.122.2.153":4,"157.109.223.109":4,"241.27.68.221":4,"77.16.23.245":4,"138.167.0.250":4,"61.231.186.42":4,"144.243.234.36":4,"168.212.76.13":4,"89.88.207.157":4,"111.226.61.72":4,"147.113.115.127":4,"64.39.36.124":4,"204.46.189.35":4,"179.5.233.205":4,"69.54.70.227":4,"37.27.12.58":4,"35.190.108.23":4,"104.74.162.127":4,"146.254.139.184":4,"5.235.151.52":4,"166.155.191.137":4,"186.254.170.103":4,"179.112.76.153":4,"129.194.200.137":4,"3.96.0.197":4,"199.141.80.189":4,"198.162.123.49":4,"168.17.86.227":4,"36.28.27.113":4,"91.88.254.84":4,"202.244.149.5":4,"145.205.59.64":4,"108.197.199.159":4,"14.116.162.170":4,"119.192.107.205":4,"126.3.98.122":4,"103.61.236.206":4,"90.83.176.104":4,"168.10.110.219":4,"152.173.185.114":4,"181.118.107.178":4,"120.199.30.66":4,"51.43.99.199":4,"30.188.32.168":4,"51.88.53.168":4,"246.231.236.214":4,"68.244.2.236":4,"47.136.28.241":4,"132.57.45.224":4,"64.149.169.88":4,"2.48.0.53":4,"55.157.55.147":4,"76.139.49.29":4,"27.174.239.40":4,"1.0.28.255":4,"41.247.158.171":4,"43.253.224.58":4,"28.227.181.38":4,"59.234.83.69":4,"54.53.128.120":4,"151.183.88.222":4,"21.199.151.202":4,"120.31.77.170":4,"51.208.251.5":4,"32.182.170.36":4,"33.187.200.54":4,"100.221.72.22":4,"132.18.136.179":4,"159.47.59.71":4,"18.141.53.254":4,"65.253.44.125":4,"109.194.200.250":4,"101.204.51.104":4,"90.186.5.132":4,"132.161.162.16":4,"118.40.23.129":4,"253.248.114.56":4,"189.77.70.175":4,"197.83.32.198":4,"240.227.2.105":4,"207.205.80.158":4,"190.197.56.2":4,"113.199.44.125":4,"35.61.192.228":4,"67.252.224.81":4,"24.65.161.126":4,"165.125.211.190":4,"73.153.55.94":4,"221.147.47.252":4,"189.153.248.42":4,"103.23.239.118":4,"169.128.220.50":4,"163.227.129.165":4,"13.226.43.84":4,"121.103.124.242":4,"170.76.163.77":4,"121.222.103.46":4,"98.129.239.107":4,"211.115.103.80":4,"253.31.171.94":4,"59.245.37.2":4,"160.41.7.239":4,"46.112.18.58":4,"71.89.115.101":4,"39.82.92.21":4,"197.69.30.44":4,"177.215.76.45":4,"174.192.72.226":4,"44.205.246.123":4,"135.85.229.16":4,"132.143.116.207":4,"98.156.69.201":4,"254.193.134.120":4,"9.78.38.219":4,"182.10.240.55":4,"78.41.205.20":4,"114.136.139.113":4,"150.122.206.54":4,"129.102.99.149":4,"217.216.84.110":4,"202.128.84.33":4,"24.59.136.169":4,"68.121.136.59":4,"88.205.194.199":4,"91.221.129.244":4,"163.186.68.10":4,"78.159.223.107":4,"75.228.49.62":4,"217.170.228.2":4,"197.46.108.42":4,"13.211.255.132":4,"14.79.242.217":4,"252.22.196.107":4,"149.65.43.201":4,"113.255.211.158":4,"92.140.63.109":4,"152.34.21.89":4,"136.226.75.106":4,"5.174.170.254":4,"131.79.184.81":4,"255.25.43.205":4,"79.145.75.153":4,"242.36.206.112":4,"186.79.161.83":4,"219.168.197.65":4,"7.3.65.17":4,"37.81.36.189":4,"199.146.249.166":4,"157.225.57.11":4,"248.231.244.246":4,"65.184.174.2":4,"213.183.215.101":4,"184.113.219.143":4,"254.252.169.147":4,"196.136.234.7":4,"3.20.234.161":4,"19.205.168.13":4,"222.114.31.109":4,"147.157.53.189":4,"196.250.79.40":4,"107.241.215.24":4,"242.228.32.60":4,"45.7.100.174":4,"215.109.179.156":4,"71.18.236.184":4,"183.43.31.84":4,"31.209.15.208":4,"1.83.144.231":4,"17.150.208.49":4,"24.6.43.100":4,"72.84.70.10":4,"114.116.25.153":4,"85.242.134.9":4,"48.39.74.180":4,"188.97.94.119":4,"125.23.91.247":4,"180.8.71.80":4,"190.59.219.63":4,"100.247.75.97":4,"130.225.56.192":4,"3.117.84.225":4,"248.20.41.32":4,"110.167.191.250":4,"134.42.113.42":4,"200.110.210.191":4,"51.134.175.134":4,"121.202.240.35":4,"120.193.93.221":4,"108.8.210.240":4,"137.74.33.199":4,"172.72.172.89":4,"255.163.70.18":4,"199.94.38.161":4,"168.15.205.28":4,"18.214.159.183":4,"75.109.127.113":4,"217.246.236.75":4,"18.89.103.244":4,"162.21.106.117":4,"156.134.183.94":4,"19.167.245.153":4,"251.23.208.165":4,"107.40.180.36":4,"105.164.150.235":4,"248.107.30.213":4,"190.123.172.75":4,"144.104.171.25":4,"201.227.157.76":4,"154.44.52.219":4,"34.98.243.64":4,"124.11.15.130":4,"109.33.12.214":4,"62.5.219.94":4,"69.179.4.211":4,"170.4.3.4":4,"166.231.176.13":4,"146.224.7.214":4,"117.14.156.137":4,"170.5.79.107":4,"157.122.135.77":4,"162.93.141.4":4,"111.161.155.65":4,"63.19.66.137":4,"29.98.101.42":4,"165.35.180.197":4,"2.11.243.136":4,"124.171.249.210":4,"81.178.185.25":4,"124.10.92.122":4,"59.139.105.216":4,"45.226.128.110":4,"166.43.34.21":4,"130.168.143.218":4,"31.74.123.239":4,"211.70.0.13":4,"111.136.66.152":4,"4.81.93.216":4,"187.183.110.197":4,"166.5.10.164":4,"245.168.156.33":4,"2.42.213.247":4,"143.11.96.11":4,"183.251.206.118":4,"54.136.91.229":4,"16.252.202.119":4,"75.87.127.133":4,"214.8.31.176":4,"104.50.37.54":4,"7.197.42.56":4,"63.166.82.26":4,"253.137.97.117":4,"143.41.158.248":4,"188.105.255.67":4,"51.25.159.198":4,"39.161.135.8":4,"200.6.216.136":4,"139.18.172.182":4,"137.138.151.196":4,"209.103.109.142":4,"39.62.204.233":4,"178.238.253.42":4,"101.181.49.252":4,"85.213.91.194":4,"60.86.64.101":4,"91.101.33.254":4,"245.208.6.13":4,"57.124.190.103":4,"64.100.96.85":4,"169.15.198.164":4,"29.163.241.202":4,"73.146.154.180":4,"190.62.47.26":4,"95.236.112.132":4,"200.245.10.228":4,"47.184.211.78":4,"150.188.2.101":4,"204.179.123.55":4,"221.222.219.8":4,"97.239.74.82":4,"208.224.44.52":4,"240.170.146.12":4,"21.112.53.114":4,"88.25.148.94":4,"103.37.63.22":4,"158.72.12.69":4,"249.240.215.136":4,"120.114.51.22":4,"41.175.88.182":4,"122.45.209.240":4,"36.33.174.170":4,"3.31.216.188":4,"99.0.187.56":4,"41.75.233.229":4,"46.217.213.0":4,"248.89.44.151":4,"149.211.250.81":4,"73.74.194.202":4,"197.237.16.3":4,"69.92.194.230":4,"252.217.81.195":4,"166.204.102.171":4,"29.237.236.243":4,"185.207.34.130":4,"159.35.92.76":4,"18.241.192.101":4,"116.113.5.206":4,"13.142.90.167":4,"170.129.12.134":4,"194.127.159.155":4,"51.145.135.127":4,"32.199.239.214":4,"205.42.182.88":4,"114.210.1.235":4,"167.127.83.36":4,"106.23.229.219":4,"164.204.108.171":4,"218.99.131.29":4,"211.72.67.190":4,"74.231.254.250":4,"70.71.185.15":4,"202.88.225.112":4,"103.176.114.241":4,"66.214.45.181":4,"94.170.109.2":4,"34.222.25.226":4,"117.153.1.109":4,"161.56.0.206":4,"138.174.224.10":4,"175.211.171.102":4,"45.19.82.42":4,"115.190.30.11":4,"18.148.94.47":4,"116.41.6.150":4,"159.152.73.81":4,"6.12.127.160":4,"89.70.44.57":4,"83.12.77.210":4,"79.63.203.4":4,"221.79.41.79":4,"125.16.184.92":4,"120.103.53.143":4,"255.205.62.23":4,"180.3.77.192":4,"207.250.206.140":4,"125.10.91.95":4,"245.201.109.94":4,"212.109.176.172":4,"121.23.180.197":4,"30.154.116.234":4,"73.206.5.224":4,"54.15.9.180":4,"40.20.177.68":4,"151.93.93.119":4,"211.29.96.173":4,"32.35.131.201":4,"13.70.203.21":4,"241.31.119.162":4,"216.75.151.202":4,"19.18.209.234":4,"4.70.233.244":4,"86.57.51.114":4,"31.196.221.196":4,"173.248.178.212":4,"112.33.125.197":4,"157.203.144.194":4,"4.44.69.176":4,"129.96.192.199":4,"206.18.79.216":4,"115.184.221.227":4,"98.184.244.7":4,"10.217.56.190":4,"27.21.83.128":4,"202.190.226.219":4,"152.113.11.141":4,"31.67.124.14":4,"13.10.102.134":4,"88.180.222.36":4,"244.98.71.38":4,"247.143.246.68":4,"11.196.249.251":4,"24.48.158.134":4,"250.194.143.156":4,"30.197.181.228":4,"19.93.149.59":4,"33.180.91.125":4,"101.77.240.172":4,"25.11.117.34":4,"50.149.167.25":4,"73.65.212.193":4,"54.163.35.141":4,"77.5.72.141":4,"101.0.135.170":4,"219.15.49.142":4,"210.208.237.222":4,"130.145.111.99":4,"221.224.9.144":4,"114.192.255.180":4,"111.223.243.189":4,"1.242.45.188":4,"15.18.218.126":4,"37.206.102.67":4,"83.153.232.86":4,"7.77.200.88":4,"172.0.18.248":4,"32.32.64.147":4,"136.191.206.231":4,"162.21.234.83":4,"255.11.50.18":4,"190.195.18.160":4,"198.134.209.122":4,"199.250.49.78":4,"222.249.216.246":4,"30.119.127.60":4,"45.58.207.247":4,"177.215.72.188":4,"246.5.212.116":4,"30.234.65.179":4,"114.174.37.18":4,"61.79.164.124":4,"156.8.24.183":4,"60.182.23.130":4,"118.88.174.238":4,"191.252.118.71":4,"53.56.118.7":4,"186.133.223.236":4,"13.230.231.175":4,"244.92.126.64":4,"177.198.12.116":4,"35.176.160.186":4,"6.60.159.90":4,"211.220.42.106":4,"126.5.224.175":4,"7.136.246.72":4,"76.30.252.103":4,"160.221.18.9":4,"67.83.84.18":4,"35.58.17.26":4,"142.193.16.10":4,"93.88.71.150":4,"142.7.13.140":4,"77.162.110.3":4,"251.106.69.224":4,"156.252.203.3":4,"50.140.65.209":4,"173.179.43.138":4,"109.185.65.237":4,"46.255.240.209":4,"136.125.106.66":4,"61.69.51.38":4,"117.41.220.105":4,"73.108.167.119":4,"66.88.195.118":4,"203.214.15.175":4,"46.17.67.26":4,"34.111.207.203":4,"2.46.148.85":4,"79.167.117.96":4,"87.22.25.167":4,"61.74.153.45":4,"87.112.216.73":4,"95.134.88.196":4,"28.75.163.113":4,"99.14.61.23":4,"95.40.3.225":4,"222.26.254.103":4,"216.81.9.168":4,"175.162.110.168":4,"173.134.170.141":4,"65.39.208.201":4,"36.139.110.5":4,"158.225.98.43":4,"8.99.15.129":4,"41.199.189.246":4,"211.160.255.69":4,"124.38.194.85":4,"125.18.188.201":4,"159.89.180.35":4,"94.70.178.244":4,"197.134.87.40":4,"152.53.164.95":4,"3.141.104.104":4,"202.228.47.131":4,"172.240.5.13":4,"220.91.106.183":4,"21.97.157.216":4,"49.39.108.98":4,"176.234.23.136":4,"153.9.141.54":4,"51.142.27.126":4,"160.88.241.215":4,"8.255.174.108":4,"213.211.189.206":4,"145.189.165.63":4,"200.150.251.197":4,"192.33.132.164":4,"33.135.228.18":4,"125.119.221.208":4,"193.210.235.23":4,"240.240.90.207":4,"92.110.217.140":4,"38.206.14.22":4,"170.97.93.160":4,"43.218.180.251":4,"30.220.17.162":4,"65.222.222.58":4,"175.69.144.108":4,"91.19.230.19":4,"49.183.174.119":4,"95.169.48.186":4,"199.220.200.147":4,"66.147.163.191":4,"100.13.237.157":4,"80.67.239.208":4,"181.177.63.92":4,"30.53.203.220":4,"28.216.39.218":4,"24.108.52.243":4,"125.216.120.55":4,"14.109.37.203":4,"93.80.196.86":4,"195.119.190.241":4,"150.140.80.65":4,"173.29.78.75":4,"67.52.11.54":4,"90.227.136.97":4,"203.241.115.109":4,"47.41.236.228":4,"247.124.81.76":4,"108.206.74.195":4,"28.117.37.9":4,"109.65.196.0":4,"190.0.148.118":4,"58.104.4.20":4,"92.186.46.15":4,"66.89.119.109":4,"108.122.81.213":4,"85.233.166.25":4,"179.63.22.66":4,"95.213.185.188":4,"7.201.170.126":4,"120.24.205.91":4,"3.177.201.146":4,"84.110.93.235":4,"65.71.173.235":4,"159.128.177.64":4,"136.101.224.188":4,"166.195.208.126":4,"99.34.46.120":4,"194.216.21.143":4,"125.251.190.233":4,"219.24.131.126":4,"58.197.194.160":4,"18.66.216.79":4,"182.214.116.215":4,"36.117.173.140":4,"162.61.179.191":4,"222.250.54.238":4,"152.9.51.7":4,"111.136.179.39":4,"65.161.129.31":4,"82.240.45.2":4,"136.131.59.91":4,"26.138.181.208":4,"156.77.0.77":4,"117.107.178.112":4,"191.222.36.178":4,"99.152.71.151":4,"151.21.83.242":4,"136.98.110.62":4,"72.36.68.221":4,"242.93.117.158":4,"85.92.229.221":4,"101.102.225.17":4,"113.64.122.83":4,"59.38.114.115":4,"169.140.228.18":4,"66.211.91.87":4,"85.142.152.119":4,"193.164.252.97":4,"7.55.191.168":4,"193.165.148.94":4,"35.55.196.78":4,"26.16.119.86":4,"123.195.101.255":4,"114.140.40.138":4,"138.204.230.218":4,"187.247.154.63":4,"119.67.107.135":4,"168.180.206.122":4,"159.146.136.108":4,"8.152.156.217":4,"216.91.211.237":4,"171.192.246.162":4,"200.155.241.200":4,"118.63.225.123":4,"5.121.153.60":4,"17.160.159.112":4,"183.220.138.245":4,"183.63.141.83":4,"242.170.163.0":4,"143.149.36.127":4,"140.125.189.101":4,"185.65.176.78":4,"78.193.195.57":4,"132.5.125.209":4,"189.227.211.222":4,"55.105.230.66":4,"222.71.210.255":4,"211.2.70.50":4,"12.140.15.26":4,"145.202.115.192":4,"149.17.128.8":4,"13.174.18.244":4,"81.159.186.247":4,"80.149.135.159":4,"15.211.238.149":4,"115.195.243.171":4,"185.47.161.20":4,"189.243.125.123":4,"26.205.4.18":4,"49.231.44.114":4,"53.214.22.27":4,"41.226.98.179":4,"103.197.93.76":4,"112.29.65.34":4,"173.33.142.23":4,"149.237.119.22":4,"84.44.106.34":4,"78.240.28.140":4,"36.183.61.156":4,"222.109.111.121":4,"157.58.221.36":4,"89.75.162.169":4,"66.145.203.12":4,"156.202.153.187":4,"126.156.63.54":4,"253.187.17.164":4,"105.80.139.130":4,"115.166.24.62":4,"122.3.44.131":4,"180.166.42.174":4,"218.54.141.1":4,"251.160.192.21":4,"70.231.208.170":4,"202.195.229.151":4,"198.149.17.87":4,"124.15.18.177":4,"219.221.2.133":4,"26.95.225.154":4,"122.189.61.129":4,"86.63.148.218":4,"22.162.36.223":4,"93.97.40.73":4,"52.210.100.4":4,"134.162.14.48":4,"248.57.70.114":4,"88.2.57.217":4,"47.187.210.30":4,"77.52.211.23":4,"40.73.161.225":4,"206.231.235.123":4,"88.33.19.194":4,"166.76.37.35":4,"197.226.87.96":4,"48.173.236.115":4,"97.121.96.113":4,"69.36.166.163":4,"47.130.148.77":4,"158.204.220.123":4,"203.254.23.7":4,"83.219.153.213":4,"72.39.189.68":4,"138.238.224.235":4,"114.85.128.111":4,"24.49.252.164":4,"177.240.196.133":4,"78.90.15.50":4,"34.146.145.147":4,"24.130.16.241":4,"120.230.114.208":4,"247.127.226.75":4,"84.23.42.77":4,"66.117.26.204":4,"183.86.228.39":4,"12.77.237.74":4,"191.16.58.5":4,"10.54.48.200":4,"11.56.144.187":4,"129.152.200.141":4,"104.223.58.219":4,"139.53.219.138":4,"130.87.26.22":4,"130.160.8.123":4,"65.25.83.59":4,"203.166.40.71":4,"122.204.35.180":4,"61.211.203.30":4,"91.250.115.193":4,"188.106.116.199":4,"160.45.22.197":4,"121.206.6.201":4,"49.70.2.179":4,"150.177.162.229":4,"85.103.89.154":4,"32.13.62.234":4,"20.138.246.153":4,"17.92.184.2":4,"146.135.78.32":4,"205.109.151.132":4,"216.139.252.150":4,"135.179.120.221":4,"74.241.68.196":4,"117.131.196.208":4,"144.176.24.174":4,"140.117.88.97":4,"193.218.247.126":4,"132.50.139.81":4,"106.244.21.147":4,"98.25.24.68":4,"37.120.50.108":4,"187.3.225.84":4,"45.218.177.132":4,"178.95.86.30":4,"137.75.121.169":4,"21.122.95.183":4,"101.22.15.187":4,"106.80.127.86":4,"35.141.0.35":4,"192.188.36.134":4,"70.206.185.161":4,"122.215.221.183":4,"115.149.81.153":4,"148.5.61.217":4,"159.12.232.8":4,"69.114.111.4":4,"164.167.216.67":4,"208.33.86.204":4,"39.241.154.134":4,"77.217.228.185":4,"30.101.222.137":4,"161.179.234.82":4,"143.247.54.203":4,"154.77.233.23":4,"186.135.170.199":4,"6.45.71.73":4,"57.235.225.125":4,"187.220.198.43":4,"178.170.15.34":4,"130.15.107.69":4,"154.125.49.206":4,"38.65.27.247":4,"221.51.67.7":4,"219.145.7.83":4,"53.13.108.157":4,"211.98.95.20":4,"203.23.196.221":4,"7.251.89.23":4,"139.167.249.39":4,"28.36.33.53":4,"172.173.20.85":4,"216.165.239.139":4,"30.48.169.195":4,"42.74.146.77":4,"193.173.161.218":4,"53.232.17.78":4,"92.11.54.170":4,"9.217.106.127":4,"6.208.48.67":4,"208.17.9.30":4,"35.230.143.149":4,"95.180.247.223":4,"171.185.246.151":4,"137.141.56.83":4,"223.191.114.206":4,"67.39.216.90":4,"172.42.54.187":4,"182.190.112.156":4,"22.118.159.109":4,"63.231.18.225":4,"53.166.101.73":4,"112.147.254.156":4,"159.101.128.203":4,"101.68.189.43":4,"57.114.108.57":4,"189.45.9.226":4,"34.147.33.10":4,"191.106.73.56":4,"89.144.217.31":4,"186.35.14.56":4,"130.202.165.188":4,"204.157.22.197":4,"245.92.102.212":4,"122.190.112.211":4,"26.47.90.202":4,"21.160.233.115":4,"129.220.18.9":4,"107.190.200.125":4,"89.213.204.194":4,"61.3.86.137":4,"137.200.71.217":4,"114.209.241.52":4,"121.20.66.26":4,"164.157.140.37":4,"58.68.109.45":4,"242.65.151.136":4,"248.171.201.190":4,"108.98.12.73":4,"191.64.31.119":4,"198.141.96.67":4,"80.8.204.236":4,"137.43.228.71":4,"134.90.127.220":4,"115.27.246.56":4,"3.36.249.92":4,"115.186.102.210":4,"119.18.34.44":4,"3.195.151.236":4,"132.38.72.254":4,"160.210.163.0":4,"90.14.105.233":4,"6.157.119.252":4,"31.253.102.2":4,"50.188.12.9":4,"134.59.40.198":4,"75.123.255.148":4,"35.167.249.234":4,"18.112.46.144":4,"97.153.102.71":4,"185.138.128.234":4,"216.114.12.8":4,"46.80.42.173":4,"43.248.170.166":4,"218.129.53.244":4,"48.226.155.81":4,"202.247.210.17":4,"109.19.96.117":4,"181.79.198.189":4,"210.217.93.123":4,"55.39.114.30":4,"192.114.102.194":4,"1.161.13.236":4,"12.198.90.230":4,"142.159.215.248":4,"172.120.220.28":4,"189.227.224.178":4,"95.254.56.170":4,"79.1.77.250":4,"6.185.128.181":4,"208.177.86.165":4,"91.96.134.173":4,"156.234.227.23":4,"198.23.172.2":4,"246.66.113.41":4,"110.27.90.150":4,"157.112.203.254":4,"155.52.74.191":4,"51.60.122.211":4,"215.2.183.109":4,"97.4.249.110":4,"8.68.61.141":4,"220.164.104.184":4,"114.110.121.212":4,"32.108.195.167":4,"248.139.253.99":4,"173.71.248.11":4,"165.41.68.218":4,"161.77.133.2":4,"37.135.226.42":4,"7.226.146.226":4,"133.209.137.99":4,"38.27.190.167":4,"156.235.125.211":4,"62.220.47.150":4,"26.59.166.12":4,"110.63.190.18":4,"192.96.75.211":4,"144.93.202.200":4,"87.46.98.3":4,"42.172.138.71":4,"45.195.84.17":4,"71.158.241.92":4,"130.25.42.60":4,"154.59.222.56":4,"174.203.193.27":4,"99.108.206.2":4,"80.114.61.70":4,"27.97.155.117":4,"19.225.7.9":4,"29.185.190.64":4,"96.253.90.65":4,"218.186.71.155":4,"73.224.224.148":4,"78.37.125.134":4,"120.122.89.209":4,"88.44.74.226":4,"81.34.215.114":4,"33.162.171.9":4,"210.25.146.74":4,"195.188.154.144":4,"204.152.219.15":4,"70.212.122.24":4,"56.199.91.212":4,"176.71.114.231":4,"4.191.116.218":4,"192.157.20.31":4,"142.103.17.190":4,"13.76.138.30":4,"134.213.187.204":4,"64.108.98.104":4,"155.94.181.200":4,"87.171.206.47":4,"184.39.225.86":4,"187.230.228.78":4,"73.110.238.46":4,"107.130.163.185":4,"19.151.223.21":4,"12.43.40.94":4,"84.239.8.168":4,"64.213.87.158":4,"153.60.91.180":4,"242.28.110.255":4,"78.160.15.80":4,"17.195.168.175":4,"170.173.78.199":4,"103.69.164.59":4,"23.14.225.26":4,"84.43.204.157":4,"172.46.126.18":4,"200.204.208.34":4,"16.156.38.47":4,"189.120.73.95":4,"27.217.39.155":4,"47.92.21.79":4,"143.131.114.211":4,"29.22.145.192":4,"153.194.224.213":4,"24.241.41.66":4,"33.80.65.250":4,"92.223.137.235":4,"120.208.152.251":4,"75.121.151.2":4,"215.235.148.5":4,"151.230.134.71":4,"136.155.3.90":4,"163.101.17.82":4,"27.222.103.42":4,"203.156.144.46":4,"118.69.102.132":4,"180.123.141.214":4,"62.246.35.248":4,"88.18.188.80":4,"139.68.189.206":4,"123.14.196.68":4,"187.92.228.108":4,"39.120.31.132":4,"129.225.226.252":4,"8.167.146.235":4,"200.52.76.13":4,"110.255.120.213":4,"170.181.159.118":4,"223.129.66.33":4,"223.148.52.39":4,"95.6.51.75":4,"72.35.245.1":4,"202.124.2.169":4,"40.112.93.130":4,"111.37.137.148":4,"194.195.33.151":4,"133.165.27.182":4,"101.14.125.121":4,"40.61.182.200":4,"138.168.215.78":4,"162.83.227.3":4,"105.129.123.185":4,"165.228.159.38":4,"52.134.13.68":4,"115.237.98.92":4,"24.203.46.130":4,"62.24.238.71":4,"32.125.210.216":4,"110.89.248.239":4,"59.142.66.171":4,"53.252.255.168":4,"160.210.97.170":4,"49.4.217.170":4,"15.83.87.3":4,"173.149.232.220":4,"161.248.151.10":4,"44.244.23.206":4,"189.237.39.90":4,"77.246.99.109":4,"184.242.179.218":4,"151.209.40.100":4,"52.215.226.147":4,"44.106.45.19":4,"246.154.69.80":4,"116.177.254.179":4,"72.111.37.107":4,"93.115.120.241":4,"27.163.5.78":4,"51.30.236.50":4,"85.41.188.34":4,"93.248.71.230":4,"88.154.213.70":4,"169.106.249.40":4,"168.66.238.147":4,"201.5.8.24":4,"199.163.27.251":4,"58.200.106.107":4,"89.96.54.188":4,"3.195.67.100":4,"241.75.195.123":4,"215.93.168.78":4,"143.21.101.226":4,"79.23.67.79":4,"9.186.150.130":4,"45.146.220.50":4,"48.131.245.16":4,"44.77.79.85":4,"21.25.28.240":4,"142.56.213.232":4,"53.147.255.186":4,"87.223.128.62":4,"172.32.111.49":4,"52.7.212.48":4,"137.223.135.138":4,"110.7.54.72":4,"49.2.127.29":4,"103.67.241.9":4,"253.241.90.42":4,"143.27.130.117":4,"55.72.17.36":4,"192.80.183.156":4,"113.6.135.26":4,"35.67.58.63":4,"37.169.255.106":4,"169.18.115.162":4,"11.161.253.99":4,"146.136.168.199":4,"141.220.183.89":4,"128.236.210.226":4,"24.36.78.190":4,"132.190.63.115":4,"160.123.31.54":4,"185.196.29.134":4,"214.241.247.188":4,"24.118.127.63":4,"189.135.74.250":4,"13.116.31.80":4,"16.179.74.151":4,"206.46.51.249":4,"197.154.129.16":4,"130.43.79.214":4,"42.49.117.225":4,"88.169.192.46":4,"25.104.217.11":4,"113.78.181.189":4,"75.6.89.43":4,"93.120.225.202":4,"135.7.149.164":4,"54.46.123.88":4,"208.114.57.25":4,"178.226.188.167":4,"48.100.36.68":4,"255.33.25.21":4,"136.78.135.152":4,"218.142.185.85":4,"184.4.121.233":4,"61.77.186.11":4,"207.216.252.166":4,"77.159.79.205":4,"141.203.152.135":4,"147.101.28.66":4,"144.98.112.140":4,"222.204.128.74":4,"178.173.41.0":4,"115.228.225.57":4,"103.9.119.94":4,"193.65.97.123":4,"80.11.44.0":4,"75.203.55.136":4,"20.137.45.157":4,"97.89.7.9":4,"88.232.114.108":4,"113.8.137.186":4,"24.98.86.182":4,"111.242.143.88":4,"166.178.134.186":4,"165.34.231.11":4,"119.68.47.90":4,"46.8.17.94":4,"102.106.224.182":4,"105.72.106.89":4,"206.125.251.9":4,"100.69.208.10":4,"37.57.55.91":4,"25.135.178.172":4,"35.51.175.95":4,"189.144.160.113":4,"59.245.55.84":4,"145.234.166.1":4,"182.75.48.234":4,"100.8.107.206":4,"102.44.248.4":4,"113.19.73.2":4,"5.92.94.32":4,"177.30.223.48":4,"213.218.248.225":4,"182.73.185.58":4,"76.63.231.166":4,"142.141.173.33":4,"222.40.41.52":4,"3.98.180.103":4,"210.18.124.224":4,"3.135.91.128":4,"74.57.73.80":4,"178.206.159.207":4,"129.120.21.89":4,"97.86.238.41":4,"23.214.189.216":4,"36.132.123.44":4,"23.226.150.69":4,"185.239.93.222":4,"133.68.155.20":4,"52.174.15.205":4,"42.126.199.27":4,"122.102.88.45":4,"186.209.83.25":4,"13.239.149.253":4,"5.34.106.26":4,"23.1.138.23":4,"213.94.167.249":4,"11.15.27.106":4,"115.244.50.112":4,"192.136.108.7":4,"197.225.180.108":4,"110.190.222.137":4,"136.187.199.227":4,"82.178.115.72":4,"147.90.48.192":4,"24.52.36.108":4,"62.56.187.156":4,"25.40.153.60":4,"5.28.180.217":4,"23.166.170.63":4,"87.242.103.245":4,"34.64.189.244":4,"198.81.47.150":4,"189.166.0.26":4,"146.187.5.32":4,"19.111.29.234":4,"143.200.63.14":4,"109.237.87.162":4,"209.193.91.4":4,"12.130.180.103":4,"136.145.144.233":4,"59.136.187.183":4,"66.16.1.225":4,"91.95.203.143":4,"104.61.19.54":4,"91.15.68.183":4,"114.107.55.220":4,"123.28.178.155":4,"50.3.229.199":4,"251.216.77.41":4,"202.140.157.106":4,"120.140.36.155":4,"189.241.245.117":4,"46.209.67.25":4,"19.46.37.69":4,"189.198.109.236":4,"18.202.13.5":4,"158.1.192.65":4,"94.103.180.61":4,"189.70.238.55":4,"22.74.222.142":4,"27.72.117.150":4,"143.14.14.42":4,"179.38.60.244":4,"198.210.247.210":4,"193.126.135.51":4,"215.38.86.86":4,"138.83.34.174":4,"109.5.148.53":4,"121.169.70.178":4,"6.62.191.254":4,"15.40.96.205":4,"131.186.78.177":4,"214.82.137.12":4,"40.40.151.140":4,"214.3.197.229":4,"208.30.12.255":4,"195.107.46.178":4,"58.38.33.8":4,"89.234.91.200":4,"167.49.206.233":4,"65.89.244.106":4,"193.93.183.202":4,"33.16.88.229":4,"47.176.197.170":4,"194.0.153.188":4,"43.235.134.77":4,"74.138.212.128":4,"79.77.24.51":4,"92.45.136.4":4,"31.178.122.17":4,"58.246.244.125":4,"185.124.153.33":4,"114.34.225.20":4,"148.241.224.140":4,"221.68.213.225":4,"28.61.214.210":4,"63.209.100.136":4,"209.252.35.225":4,"114.228.210.158":4,"14.45.201.180":4,"72.97.39.48":4,"33.21.231.199":4,"2.221.245.93":4,"136.70.70.212":4,"189.238.202.52":4,"75.187.184.26":4,"163.45.232.116":4,"128.48.179.187":4,"248.225.235.22":4,"113.134.248.58":4,"181.205.74.30":4,"67.60.51.143":4,"142.44.146.80":4,"179.56.114.18":4,"155.224.227.10":4,"251.95.137.238":4,"170.69.140.88":4,"115.242.58.215":4,"200.39.19.159":4,"112.166.151.93":4,"166.139.202.249":4,"120.49.89.162":4,"35.28.12.174":4,"241.12.13.155":4,"75.35.137.54":4,"130.151.249.69":4,"216.25.42.171":4,"6.197.201.48":4,"188.138.144.120":4,"177.87.218.75":4,"202.134.120.144":4,"36.54.174.146":4,"44.5.164.19":4,"213.231.254.24":4,"18.110.216.60":4,"3.137.164.79":4,"54.183.82.17":4,"99.151.64.115":4,"15.134.229.197":4,"214.82.160.124":4,"82.54.42.26":4,"114.20.129.80":4,"29.56.172.45":4,"45.176.183.171":4,"68.115.234.202":4,"149.16.86.60":4,"27.57.164.235":4,"1.7.147.58":4,"153.34.114.64":4,"166.228.153.148":4,"139.166.26.157":4,"165.141.104.95":4,"56.128.77.213":4,"10.195.102.245":4,"201.194.239.54":4,"199.52.147.234":4,"252.171.127.74":4,"35.97.207.196":4,"65.129.67.228":4,"24.232.127.24":4,"137.120.111.15":4,"123.157.26.74":4,"7.211.46.214":4,"29.218.176.140":4,"112.217.95.88":4,"217.154.130.173":4,"109.227.28.206":4,"30.91.167.13":4,"171.206.176.253":4,"55.89.160.155":4,"34.186.197.36":4,"168.94.170.82":4,"33.141.207.44":4,"206.173.75.142":4,"215.224.212.136":4,"131.78.170.242":4,"48.98.31.187":4,"23.150.45.136":4,"28.104.30.101":4,"162.51.89.86":4,"48.131.46.145":4,"219.175.45.43":4,"5.90.202.95":4,"76.2.176.3":4,"15.207.43.4":4,"159.150.225.142":4,"77.158.190.128":4,"164.64.124.7":4,"4.111.127.132":4,"215.212.50.161":4,"166.141.229.239":4,"109.171.89.130":4,"199.4.107.177":4,"150.24.222.117":4,"175.154.30.148":4,"21.19.242.104":4,"2.189.187.206":4,"133.200.252.191":4,"201.194.128.192":4,"122.180.114.75":4,"146.249.140.17":4,"100.178.191.194":4,"148.215.249.68":4,"164.117.233.195":4,"196.219.52.183":4,"4.126.65.151":4,"99.223.175.146":4,"15.218.34.52":4,"109.100.35.15":4,"186.46.52.33":4,"16.101.136.73":4,"15.48.225.130":4,"248.211.34.7":4,"209.220.255.132":4,"152.172.26.85":4,"45.251.50.251":4,"58.160.17.242":4,"166.64.77.181":4,"199.21.137.207":4,"84.92.59.202":4,"146.63.45.146":4,"183.79.83.187":4,"1.161.219.202":4,"33.202.58.95":4,"179.79.26.127":4,"1.178.24.186":4,"142.241.153.54":4,"138.19.134.240":4,"176.110.47.189":4,"249.171.61.229":4,"50.100.62.88":4,"12.78.253.41":4,"216.80.55.196":4,"41.194.128.198":4,"15.203.116.227":4,"173.24.173.168":4,"116.236.114.251":4,"51.168.144.80":4,"189.11.220.76":4,"46.202.132.170":4,"53.92.79.185":4,"40.14.162.182":4,"10.91.249.37":4,"145.179.112.252":4,"176.84.211.185":4,"120.52.226.247":4,"219.190.16.4":4,"131.59.115.200":4,"142.78.199.254":4,"8.8.39.178":4,"30.114.220.33":4,"71.75.130.71":4,"31.82.37.178":4,"162.39.96.64":4,"190.212.79.238":4,"8.68.210.63":4,"217.23.131.78":4,"201.182.227.208":4,"147.18.125.138":4,"52.217.166.19":4,"222.51.58.62":4,"203.225.52.174":4,"107.146.83.254":4,"48.126.236.208":4,"8.203.56.65":4,"90.126.168.151":4,"195.181.118.90":4,"176.216.189.148":4,"134.113.139.177":4,"38.128.244.54":4,"83.221.130.16":4,"23.52.64.87":4,"151.170.101.84":4,"53.46.200.36":4,"138.68.81.224":4,"54.63.51.219":4,"186.161.222.147":4,"16.137.242.160":4,"8.17.236.237":4,"74.54.205.210":4,"123.62.171.118":4,"85.13.138.242":4,"182.189.252.166":4,"222.144.170.160":4,"201.111.47.179":4,"122.118.194.153":4,"111.1.229.115":4,"216.149.212.15":4,"105.172.3.185":4,"196.165.0.130":4,"126.220.37.13":4,"46.16.154.134":4,"24.106.2.253":4,"104.11.111.87":4,"4.175.38.216":4,"184.104.236.0":4,"57.254.223.116":4,"223.38.119.15":4,"19.155.245.186":4,"3.19.62.169":4,"101.68.9.85":4,"74.174.19.130":4,"5.191.98.165":4,"129.134.9.77":4,"11.70.79.71":4,"63.48.235.219":4,"5.98.95.179":4,"210.4.221.52":4,"39.71.118.247":4,"138.92.248.232":4,"150.234.50.54":4,"220.50.20.35":4,"61.27.21.202":4,"173.122.66.17":4,"89.113.29.95":4,"24.101.179.227":4,"143.150.120.137":4,"69.21.56.88":4,"199.255.81.27":4,"58.152.250.159":4,"30.246.227.202":4,"196.23.69.184":4,"22.58.32.72":4,"204.245.196.181":4,"45.165.19.75":4,"34.127.202.25":4,"102.164.94.227":4,"212.68.182.164":4,"221.34.87.185":4,"117.57.11.255":4,"123.173.204.50":4,"79.118.188.155":4,"38.238.252.211":4,"33.68.214.203":4,"120.149.45.180":4,"102.21.192.222":4,"160.199.118.198":4,"172.202.187.1":4,"205.128.50.135":4,"1.219.118.12":4,"69.95.36.149":4,"50.35.70.162":4,"57.241.102.30":4,"200.79.64.33":4,"17.22.122.220":4,"94.146.207.169":4,"69.46.194.89":4,"186.113.191.221":4,"36.163.230.255":4,"92.37.74.6":4,"128.182.16.0":4,"23.88.144.66":4,"86.178.143.163":4,"71.211.136.241":4,"170.229.86.224":4,"21.209.231.119":4,"185.157.165.86":4,"15.161.141.118":4,"128.165.6.119":4,"47.93.42.0":4,"159.123.16.164":4,"201.65.21.108":4,"215.166.185.179":4,"111.33.16.167":4,"69.27.139.57":4,"222.153.240.149":4,"189.31.19.224":4,"246.186.123.33":4,"135.18.125.51":4,"35.239.82.210":4,"20.140.199.176":4,"68.158.12.192":4,"202.194.137.127":4,"156.202.96.21":4,"196.16.185.108":4,"10.198.173.252":4,"153.108.4.62":4,"14.82.92.251":4,"193.163.183.175":4,"209.101.227.3":4,"122.193.134.136":4,"112.192.201.5":4,"253.235.239.184":4,"55.223.165.24":4,"75.87.169.128":4,"35.78.187.125":4,"2.165.89.138":4,"128.79.116.135":4,"195.59.246.206":4,"85.99.151.219":4,"25.204.246.213":4,"149.195.7.9":4,"156.13.227.249":4,"207.126.97.157":4,"54.10.135.54":4,"223.104.243.73":4,"130.244.177.18":4,"240.95.142.40":4,"1.31.52.81":4,"94.243.119.184":4,"193.64.47.87":4,"64.29.247.202":4,"140.1.194.125":4,"98.49.70.198":4,"107.98.184.80":4,"9.164.177.27":4,"98.43.168.59":4,"131.72.33.224":4,"103.83.247.247":4,"221.222.77.228":4,"21.149.82.207":4,"194.87.165.66":4,"21.104.11.129":4,"161.226.108.232":4,"179.236.140.205":4,"251.248.129.182":4,"18.33.50.200":4,"243.234.78.220":4,"61.99.196.53":4,"65.104.98.29":4,"119.160.235.214":4,"48.104.13.110":4,"151.54.178.42":4,"151.110.116.75":4,"52.210.109.214":4,"22.47.240.172":4,"249.56.148.154":4,"83.213.246.184":4,"75.155.150.54":4,"98.9.191.115":4,"10.123.116.47":4,"216.192.171.194":4,"13.70.187.201":4,"207.116.86.160":4,"81.34.97.70":4,"43.193.152.172":4,"39.16.8.58":4,"58.185.197.4":4,"114.255.89.39":4,"113.1.11.255":4,"25.171.140.238":4,"107.247.207.209":4,"32.80.212.43":4,"24.225.255.41":4,"50.83.120.137":4,"173.60.90.130":4,"22.238.153.179":4,"157.124.139.151":4,"195.16.238.69":4,"82.33.58.210":4,"183.97.166.40":4,"176.196.185.54":4,"92.223.90.220":4,"71.194.20.98":4,"140.49.147.88":4,"32.189.141.61":4,"208.147.177.251":4,"88.92.246.142":4,"74.182.216.247":4,"203.206.31.45":4,"154.114.19.82":4,"65.62.184.83":4,"47.59.75.20":4,"188.138.193.20":4,"108.172.38.144":4,"121.246.174.60":4,"52.194.31.94":4,"145.83.238.94":4,"90.57.105.228":4,"98.71.69.111":4,"47.133.219.161":4,"32.119.81.71":4,"175.174.143.76":4,"140.244.242.224":4,"117.21.159.1":4,"24.251.73.36":4,"99.175.194.67":4,"58.167.28.173":4,"9.252.187.112":4,"180.132.4.71":4,"118.115.35.50":4,"10.38.146.194":4,"118.29.23.196":4,"98.181.208.155":4,"89.9.80.5":4,"143.36.80.102":4,"60.230.11.134":4,"155.234.156.152":4,"158.0.16.248":4,"215.206.47.139":4,"113.73.234.69":4,"10.85.49.143":4,"29.61.93.157":4,"93.213.183.157":4,"123.219.63.6":4,"205.125.195.83":4,"105.199.56.23":4,"2.227.126.196":4,"100.217.7.37":4,"109.188.60.163":4,"202.125.221.43":4,"40.80.13.86":4,"197.54.226.115":4,"90.39.160.35":4,"140.113.61.139":4,"39.254.115.118":4,"145.118.172.62":4,"82.108.71.118":4,"86.199.110.139":4,"134.231.193.133":4,"53.58.124.228":4,"30.223.165.204":4,"135.23.164.109":4,"109.96.138.157":4,"66.3.233.152":4,"105.118.149.0":4,"51.56.135.25":4,"23.167.225.14":4,"29.221.226.245":4,"135.193.131.179":4,"132.66.3.216":4,"117.10.127.53":4,"240.105.208.108":4,"60.42.67.152":4,"34.76.78.130":4,"42.113.239.40":4,"29.248.62.207":4,"4.206.112.160":4,"136.238.221.97":4,"171.7.125.87":4,"52.122.117.12":4,"153.208.8.127":4,"77.78.159.181":4,"179.39.24.49":4,"160.98.224.105":4,"104.0.51.242":4,"150.19.193.141":4,"8.180.231.149":4,"37.158.69.55":4,"32.193.99.23":4,"45.28.40.3":4,"152.218.78.53":4,"60.168.214.6":4,"196.207.145.95":4,"51.224.165.150":4,"43.150.206.33":4,"28.9.41.151":4,"18.79.69.153":4,"215.96.71.76":4,"60.37.185.233":4,"61.123.77.51":4,"1.196.30.249":4,"48.230.205.64":4,"201.230.70.96":4,"190.158.93.247":4,"93.126.225.25":4,"151.81.226.166":4,"3.164.7.15":4,"42.230.4.238":4,"144.74.233.18":4,"33.116.59.126":4,"90.120.16.24":4,"193.210.189.6":4,"128.39.102.208":4,"81.231.148.9":4,"196.119.244.74":4,"94.244.240.183":4,"10.250.179.237":4,"161.134.86.13":4,"30.2.241.157":4,"132.145.55.151":4,"10.118.141.103":4,"84.26.231.2":4,"3.98.173.177":4,"36.95.87.19":4,"216.168.166.154":4,"55.55.133.59":4,"20.11.53.147":4,"95.217.69.180":4,"88.201.155.95":4,"146.178.70.227":4,"124.246.46.130":4,"91.228.47.64":4,"104.0.247.243":4,"42.114.36.107":4,"186.34.70.133":4,"28.230.122.123":4,"119.110.84.131":4,"68.127.104.208":4,"93.208.92.175":4,"208.52.89.231":4,"77.16.51.8":4,"66.64.122.161":4,"169.248.201.16":4,"183.59.162.54":4,"149.247.42.157":4,"161.56.255.119":4,"241.232.74.47":4,"201.188.91.250":4,"180.89.106.44":4,"156.87.31.68":4,"65.2.156.145":4,"126.159.190.171":4,"221.154.213.234":4,"39.174.65.28":4,"82.169.127.189":4,"168.172.238.179":4,"219.98.239.169":4,"214.54.146.150":4,"167.244.231.5":4,"39.5.58.161":4,"28.145.3.185":4,"174.196.31.214":4,"74.203.144.23":4,"193.237.210.108":4,"221.200.220.170":4,"84.75.118.197":4,"91.133.15.111":4,"60.107.184.45":4,"155.107.2.124":4,"191.212.27.126":4,"9.101.8.9":4,"92.6.33.81":4,"136.188.154.80":4,"9.98.145.86":4,"51.84.113.182":4,"140.14.25.165":4,"122.60.105.120":4,"193.148.143.4":4,"161.43.222.65":4,"66.83.119.72":4,"52.243.198.49":4,"90.238.102.24":4,"196.5.132.140":4,"75.22.137.246":4,"64.51.96.235":4,"31.26.128.180":4,"195.233.151.28":4,"41.205.46.26":4,"171.232.60.33":4,"98.3.203.95":4,"75.203.79.212":4,"52.186.19.124":4,"47.222.49.160":4,"223.5.68.67":4,"87.57.148.168":4,"111.53.32.26":4,"73.4.45.194":4,"174.101.101.201":4,"32.73.251.172":4,"188.1.185.56":4,"52.1.72.26":4,"223.154.247.97":4,"107.13.133.127":4,"116.8.99.239":4,"93.150.88.84":4,"211.187.42.37":4,"95.51.28.238":4,"67.193.104.235":4,"25.180.52.72":4,"162.226.172.4":4,"155.32.247.242":4,"64.254.91.67":4,"32.20.58.36":4,"149.252.33.32":4,"198.248.152.227":4,"11.14.249.116":4,"191.97.27.147":4,"134.226.208.43":4,"150.121.182.219":4,"125.141.2.113":4,"110.222.82.31":4,"197.156.12.249":4,"150.255.17.82":4,"114.225.71.116":4,"120.26.171.48":4,"14.15.192.252":4,"159.249.0.139":4,"102.148.4.105":4,"24.214.139.156":4,"191.179.126.121":4,"8.147.40.16":4,"208.160.212.57":4,"68.7.90.162":4,"171.91.81.59":4,"160.165.167.43":4,"66.253.209.34":4,"61.161.207.224":4,"115.119.152.87":4,"156.157.17.49":4,"69.173.63.192":4,"39.100.75.71":4,"170.223.204.243":4,"117.191.52.81":4,"19.71.26.126":4,"207.121.211.33":4,"44.166.242.202":4,"59.19.196.250":4,"189.215.232.239":4,"50.41.243.194":4,"114.56.172.93":4,"40.167.85.210":4,"150.177.27.211":4,"27.212.171.175":4,"27.240.235.57":4,"118.196.114.44":4,"167.156.70.11":4,"21.169.231.84":4,"198.212.218.220":4,"204.216.116.104":4,"17.246.119.66":4,"210.231.49.230":4,"2.28.96.43":4,"13.21.209.174":4,"55.51.62.255":4,"48.176.54.30":4,"133.154.177.180":4,"62.229.199.255":4,"186.24.112.27":4,"185.49.160.67":4,"131.127.39.136":4,"102.199.135.135":4,"150.27.183.87":4,"242.96.155.159":4,"29.88.194.216":4,"82.123.36.50":4,"73.50.67.9":4,"176.182.40.32":4,"172.137.249.99":4,"198.102.165.212":4,"119.244.215.154":4,"206.188.163.95":4,"179.29.170.109":4,"177.49.114.161":4,"180.208.21.244":4,"171.244.30.230":4,"198.82.227.23":4,"205.43.50.203":4,"219.169.97.102":4,"59.182.230.218":4,"134.74.82.109":4,"186.120.52.91":4,"141.180.120.17":4,"91.246.55.117":4,"220.23.142.88":4,"66.124.15.91":4,"6.41.232.125":4,"219.95.50.132":4,"141.159.43.88":4,"153.208.251.74":4,"56.138.56.83":4,"191.13.69.79":4,"203.254.222.59":4,"71.109.254.1":4,"60.3.173.200":4,"73.159.227.185":4,"219.97.196.143":4,"208.237.202.158":4,"170.254.253.12":4,"195.62.167.245":4,"3.41.115.51":4,"139.203.156.145":4,"2.15.217.177":4,"8.77.141.141":4,"92.255.197.249":4,"149.208.156.189":4,"34.192.172.220":4,"47.88.149.226":4,"30.203.147.181":4,"219.224.74.178":4,"149.173.32.85":4,"9.182.254.250":4,"97.112.23.141":4,"215.14.217.229":4,"142.87.56.25":4,"19.30.49.140":4,"134.127.67.31":4,"147.81.142.209":4,"74.78.202.240":4,"120.17.181.126":4,"255.255.255.255":4,"10.49.22.22":4,"fe80::347c:c9cc:1a3f:f002":4,"fe80::bd09:759a:a68f:4252":4,"2405:6e00:10d4:c800:20c:29ff:fe28:72d1":4,"fe80::2530:6f6:19fd:103b":4,"fe80::20c:29ff:fe61:7ed2":4}

## Run the experimental grid

In [ ]:
import subprocess, sys
JOBS = [(p,'bin',s) for p in ['P1','P2','P3'] for s in range(3)] + [('P4','bin',f) for f in range(5)] \
     + [('P3','multi',s) for s in range(3)] + [('P4','multi',f) for f in range(3)]
for p,t,s in JOBS:
    out = f"{os.environ['RESULTS_DIR']}/{p}_{t}_{s}.json"
    if os.path.exists(out) and '"done": true' in open(out).read(): continue
    r = subprocess.run([sys.executable,'run.py',p,t,str(s)], capture_output=True, text=True)
    print('\n'.join(l for l in r.stdout.splitlines() if l.startswith('P')) or r.stderr[-2000:])

In [ ]:
for fam in ['ransomware','backdoor','mitm','scanning','xss','password','injection','ddos','dos']:
    r = subprocess.run([sys.executable,'zeroday.py',fam,'0'], capture_output=True, text=True)
    print(r.stdout[-1500:] or r.stderr[-1500:])

## Aggregate tables, statistics and figures

In [ ]:
r = subprocess.run([sys.executable,'aggregate.py'], capture_output=True, text=True); print(r.stdout, r.stderr[-1500:])
import json; agg = json.load(open('/kaggle/working/agg.json'))
rows=[]
for p,d in agg['binary'].items():
    for m,v in d.items(): rows.append(dict(protocol=p, model=m, **{k: f"{v[k]['mean']:.4f}±{v[k]['sd']:.4f}" for k in ['mcc','f1_macro','pr_auc','fpr','recall']}))
display(pd.DataFrame(rows)); print(json.dumps(agg['stats_mcc'], indent=1))

In [ ]:
from IPython.display import Image, display
for f in sorted(glob.glob('/kaggle/working/fig/*.png')): print(f); display(Image(f, width=900))